In [ ]:
# ============================================================
# scraper_kompas_simple.py - VERSI DENGAN TAHUN FLEKSIBEL
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus

# ================================================================
# KONFIGURASI - UBAH TAHUN DI SINI
# ================================================================
TARGET_YEAR = None  # Ganti ke tahun yang ada datanya
# Atau set ke None untuk ambil SEMUA tahun
# TARGET_YEAR = None

OUTPUT_FILE = f"kompas_pdp_{TARGET_YEAR if TARGET_YEAR else 'all'}.csv"
MAX_PAGES = 5

KEYWORDS = [
    "perlindungan data pribadi",
    "UU PDP",
    "kebocoran data",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI
# ================================================================

def get_date_and_year(soup):
    """Ambil tanggal dan tahun dari artikel"""
    try:
        date_tag = soup.find("div", class_="read__time")
        if date_tag:
            text = date_tag.text.strip()
            parts = text.split()
            if len(parts) >= 3:
                day = parts[0]
                month = parts[1]
                year = parts[2]

                months = {"Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
                         "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12}

                if month in months:
                    date_str = f"{year}-{months[month]:02}-{int(day):02}"
                    return date_str, int(year)
    except:
        pass
    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE
# ================================================================

all_data = []
year_stats = {}  # Untuk lihat distribusi tahun

for keyword in KEYWORDS:
    print(f"\n🔍 Mencari: {keyword}")
    encoded = quote_plus(keyword)

    for page in range(1, MAX_PAGES + 1):
        url = f"https://search.kompas.com/search?q={encoded}&page={page}"

        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            soup = BeautifulSoup(resp.text, "html.parser")

            links = soup.find_all("a", href=lambda x: x and "/read/" in x)

            if not links:
                break

            print(f"  Halaman {page}: {len(links)} artikel")

            for link in links:
                try:
                    title = link.text.strip()
                    article_url = link["href"]

                    time.sleep(random.uniform(0.3, 0.8))
                    detail = requests.get(article_url, headers=HEADERS, timeout=10)
                    soup_detail = BeautifulSoup(detail.text, "html.parser")

                    date_str, year = get_date_and_year(soup_detail)

                    # Statistik tahun
                    if year:
                        year_stats[year] = year_stats.get(year, 0) + 1

                    # Filter tahun (jika TARGET_YEAR ditentukan)
                    if TARGET_YEAR and year != TARGET_YEAR:
                        continue

                    content = get_content(soup_detail)

                    if content:
                        all_data.append({
                            "judul": title,
                            "tanggal": date_str,
                            "tahun": year,
                            "url": article_url,
                            "isi": content[:500]
                        })
                        print(f"    ✓ {title[:60]} ({year})")

                except Exception as e:
                    continue

        except Exception as e:
            print(f"  Error: {e}")
            break

    time.sleep(2)

# ================================================================
# HASIL
# ================================================================

print("\n" + "="*60)
print("STATISTIK TAHUN ARTIKEL YANG DITEMUKAN:")
print("="*60)
for year in sorted(year_stats.keys(), reverse=True):
    print(f"  {year}: {year_stats[year]} artikel")

print("\n" + "="*60)
if all_data:
    df = pd.DataFrame(all_data)
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"✅ {len(df)} artikel disimpan ke {OUTPUT_FILE}")
    print("\nSample 5 artikel terbaru:")
    print(df[['judul', 'tanggal']].head())
else:
    print(f"❌ Tidak ada artikel tahun {TARGET_YEAR if TARGET_YEAR else 'yang dipilih'}")
    print(f"\n💡 Tips: Coba ubah TARGET_YEAR ke tahun yang tersedia di statistik di atas")
    print(f"   Misal: TARGET_YEAR = 2025 atau TARGET_YEAR = None (ambil semua)")


🔍 Mencari: perlindungan data pribadi
  Halaman 1: 20 artikel
    ✓ ASN yang Sebarkan Data Pribadi Eks Pebalap F1 Rio Haryanto D (None)
    ✓ RI Rawan Bencana, Perlindungan Asuransi Masih Minim




     (None)
    ✓ Paradoks Perlindungan Anak di Era Digital




               (None)
    ✓ SPT Tahunan Orang Pribadi Harus Nihil?




                  (None)
    ✓ LPSK Putuskan Perlindungan untuk Saksi Kasus Andrie Yunus


 (None)
    ✓ Fenomena "Fragile Middle Class": Terjepit Pajak tapi Minim P (None)
    ✓ Rumah Jadi Fondasi Perlindungan Anak, Pengasuhan Bukan Tugas (None)
    ✓ Psikolog Soroti Bahaya Kecanduan Gadget, PP Tunas Jadi Perli (None)
    ✓ Kesulitan Bertemu Anak, Insanul Fahmi Akan ke Komisi Perlind (None)
    ✓ Selain Perlindungan, LPSK Biayai Pengobatan Andrie Yunus di  (None)
    ✓ Keluarga Pensiunan JICT Ermanto Usman Minta Perlindungan ke  (None)
    ✓ LPSK Beri Perlindungan di Kasus Penyiraman Air Keras Andrie  (None)
    ✓ Prabowo Susun 2 Kebijakan Perlindungan Gajah

In [ ]:
# ============================================================
# debug_date.py - Cek struktur tanggal di artikel Kompas
# ============================================================

import requests
from bs4 import BeautifulSoup

# Ambil satu artikel contoh
url = "https://www.kompas.com/tren/read/2026/03/11/091354065/paradoks-perlindungan-anak-di-era-digital"

headers = {"User-Agent": "Mozilla/5.0"}

resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.text, "html.parser")

print("=== MENCARI ELEMEN TANGGAL ===\n")

# Coba berbagai selector
selectors = [
    "div.read__time",
    "div.article-date",
    "time",
    "div.date",
    "span.date",
    "div[class*='time']",
    "div[class*='date']"
]

for selector in selectors:
    elements = soup.select(selector)
    if elements:
        print(f"Selector '{selector}':")
        for elem in elements[:2]:
            print(f"  Text: {elem.get_text(strip=True)}")
            print(f"  HTML: {elem}")
            print()

# Cari semua elemen yang mungkin berisi tanggal
print("\n=== SEMUA ELEMEN DENGAN KATA 'date' ATAU 'time' ===")
for elem in soup.find_all(class_=True):
    classes = " ".join(elem.get("class", []))
    if "date" in classes.lower() or "time" in classes.lower():
        print(f"Class: {classes}")
        print(f"Text: {elem.get_text(strip=True)[:100]}")
        print()

# Cek meta tags
print("\n=== META TAGS ===")
meta_date = soup.find("meta", {"property": "article:published_time"})
if meta_date:
    print(f"article:published_time: {meta_date.get('content')}")

meta_date2 = soup.find("meta", {"name": "pubdate"})
if meta_date2:
    print(f"pubdate: {meta_date2.get('content')}")

=== MENCARI ELEMEN TANGGAL ===

Selector 'div.read__time':
  Text: Kompas.com, 11 Maret 2026, 09:13 WIB
  HTML: <div class="read__time"><a href="https://www.kompas.com">Kompas.com</a>, 11 Maret 2026, 09:13 WIB</div>

Selector 'div[class*='time']':
  Text: Kompas.com, 11 Maret 2026, 09:13 WIB
  HTML: <div class="read__time"><a href="https://www.kompas.com">Kompas.com</a>, 11 Maret 2026, 09:13 WIB</div>

Selector 'div[class*='date']':
  Text: 26/03/2026, 20:00 WIB
  HTML: <div class="article__date">26/03/2026, 20:00 WIB</div>

  Text: 26/03/2026, 19:30 WIB
  HTML: <div class="article__date">26/03/2026, 19:30 WIB</div>


=== SEMUA ELEMEN DENGAN KATA 'date' ATAU 'time' ===
Class: read__time
Text: Kompas.com, 11 Maret 2026, 09:13 WIB

Class: article__date
Text: 26/03/2026, 20:00 WIB

Class: article__date
Text: 26/03/2026, 19:30 WIB

Class: article__date
Text: 26/03/2026, 19:00 WIB

Class: article__date
Text: 26/03/2026, 18:30 WIB

Class: article__date
Text: 26/03/2026, 18:00 WIB

Class: art

In [ ]:
# ============================================================
# scraper_kompas_working.py
# Fixed date extraction for Kompas articles
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re

# ================================================================
# KONFIGURASI
# ================================================================
TARGET_YEAR = 2020  # Change to 2025, 2024, or None for all years
OUTPUT_FILE = f"kompas_pdp_1page{TARGET_YEAR if TARGET_YEAR else 'all'}.csv"
MAX_PAGES = 1

KEYWORDS = [
    "perlindungan data pribadi",
    "RUU PDP",
    "UU PDP",
    "kebocoran data",
    "insiden kebocoran data pribadi",
    "kasus kebocoran data",
    "kasus kebocoran data marketplace",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI TANGGAL - FIXED!
# ================================================================

def extract_date_from_article(soup):
    """Extract date from article - handles both date formats"""

    # FIRST PRIORITY: Check meta tag (most reliable)
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date and meta_date.get("content"):
        # Format: 2026-03-11T02:13:54+00:00
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', meta_date["content"])
        if match:
            year, month, day = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # SECOND: Try div.read__time (format: "Kompas.com, 11 Maret 2026, 09:13 WIB")
    read_time = soup.find("div", class_="read__time")
    if read_time:
        text = read_time.get_text(strip=True)
        # Extract date like "11 Maret 2026"
        months = {
            "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
            "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
        }

        # Pattern: day month year
        match = re.search(r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+(\d{4})', text)
        if match:
            day, month_name, year = match.groups()
            if month_name in months:
                month = months[month_name]
                return f"{year}-{month:02}-{int(day):02}", int(year)

    # THIRD: Try div.article__date (format: "26/03/2026, 20:00 WIB")
    article_date = soup.find("div", class_="article__date")
    if article_date:
        text = article_date.get_text(strip=True)
        # Extract date like "26/03/2026"
        match = re.search(r'(\d{2})/(\d{2})/(\d{4})', text)
        if match:
            day, month, year = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # FOURTH: Try time tag
    time_tag = soup.find("time")
    if time_tag:
        if time_tag.get("datetime"):
            match = re.search(r'(\d{4})-(\d{2})-(\d{2})', time_tag["datetime"])
            if match:
                year, month, day = match.groups()
                return f"{year}-{month}-{day}", int(year)

    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if not content:
            content = soup.find("article")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE
# ================================================================

all_data = []
year_stats = {}

for keyword in KEYWORDS:
    print(f"\n🔍 Mencari: {keyword}")
    encoded = quote_plus(keyword)

    for page in range(1, MAX_PAGES + 1):
        url = f"https://search.kompas.com/search?q={encoded}&page={page}"

        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            soup = BeautifulSoup(resp.text, "html.parser")

            links = soup.find_all("a", href=lambda x: x and "/read/" in x)

            if not links:
                break

            print(f"  Halaman {page}: {len(links)} artikel")

            for link in links:
                try:
                    title = link.text.strip()
                    article_url = link["href"]

                    time.sleep(random.uniform(0.3, 0.8))
                    detail = requests.get(article_url, headers=HEADERS, timeout=10)
                    soup_detail = BeautifulSoup(detail.text, "html.parser")

                    # Get date using improved function
                    date_str, year = extract_date_from_article(soup_detail)

                    # Debug: show first few dates
                    if len(all_data) < 5 and year:
                        print(f"    📅 Found: {title[:50]}... -> {date_str}")

                    # Update statistics
                    if year:
                        year_stats[year] = year_stats.get(year, 0) + 1

                    # Filter by target year if specified
                    if TARGET_YEAR and year != TARGET_YEAR:
                        continue

                    content = get_content(soup_detail)

                    if content and date_str:
                        all_data.append({
                            "judul": title,
                            "tanggal": date_str,
                            "tahun": year,
                            "url": article_url,
                            "isi": content,
                        })
                        print(f"    ✓ {title[:60]} ({date_str})")

                except Exception as e:
                    # Silent fail for individual articles
                    continue

        except Exception as e:
            print(f"  Error page {page}: {e}")
            break

    time.sleep(2)

# ================================================================
# HASIL
# ================================================================

print("\n" + "="*60)
print("STATISTIK TAHUN ARTIKEL YANG DITEMUKAN:")
print("="*60)

if year_stats:
    for year in sorted(year_stats.keys(), reverse=True):
        print(f"  {year}: {year_stats[year]} artikel")
else:
    print("  Tidak ada tanggal yang berhasil diekstrak!")

print("\n" + "="*60)

if all_data:
    df = pd.DataFrame(all_data)
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"✅ {len(df)} artikel disimpan ke {OUTPUT_FILE}")
    print("\nSample 5 artikel terbaru:")
    print(df[['judul', 'tanggal']].head())

    # Show year distribution in collected data
    print("\nDistribusi tahun dalam data yang dikumpulkan:")
    year_dist = df['tahun'].value_counts().sort_index(ascending=False)
    for year, count in year_dist.items():
        print(f"  {year}: {count} artikel")
else:
    print(f"❌ Tidak ada data ditemukan untuk tahun {TARGET_YEAR if TARGET_YEAR else 'yang dipilih'}")
    if year_stats:
        print(f"\n💡 Tips: Coba ubah TARGET_YEAR ke tahun yang tersedia di statistik di atas")
        print(f"   Tersedia: {', '.join(str(y) for y in sorted(year_stats.keys(), reverse=True))}")


🔍 Mencari: perlindungan data pribadi
  Halaman 1: 20 artikel
    📅 Found: ASN yang Sebarkan Data Pribadi Eks Pebalap F1 Rio ... -> 2026-03-09
    📅 Found: RI Rawan Bencana, Perlindungan Asuransi Masih Mini... -> 2026-03-10
    📅 Found: Paradoks Perlindungan Anak di Era Digital




    ... -> 2026-03-11
    📅 Found: SPT Tahunan Orang Pribadi Harus Nihil?




       ... -> 2026-03-12
    📅 Found: LPSK Putuskan Perlindungan untuk Saksi Kasus Andri... -> 2026-03-17
    📅 Found: Fenomena "Fragile Middle Class": Terjepit Pajak ta... -> 2026-03-04
    📅 Found: Rumah Jadi Fondasi Perlindungan Anak, Pengasuhan B... -> 2026-03-01
    📅 Found: Psikolog Soroti Bahaya Kecanduan Gadget, PP Tunas ... -> 2026-03-25
    📅 Found: Kesulitan Bertemu Anak, Insanul Fahmi Akan ke Komi... -> 2026-03-24
    📅 Found: Selain Perlindungan, LPSK Biayai Pengobatan Andrie... -> 2026-03-17
    📅 Found: Keluarga Pensiunan JICT Ermanto Usman Minta Perlin... -> 2026-03-05
    📅 Found: LPSK Beri Perlindungan di Kasus Pe

In [ ]:
# ============================================================
# scraper_kompas_all_years.py
# Scrape semua halaman untuk mendapatkan distribusi tahun
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re
from collections import Counter

# ================================================================
# KONFIGURASI
# ================================================================
TARGET_YEAR = 2020  # Ambil semua tahun
OUTPUT_FILE = f"kompas_pdp_all_years.csv"
MAX_PAGES = 20  # Tambah halaman untuk dapat tahun yang lebih lama

KEYWORDS = [
    "perlindungan data pribadi",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI TANGGAL
# ================================================================

def extract_date_from_article(soup):
    """Extract date from article"""

    # Check meta tag first
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date and meta_date.get("content"):
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', meta_date["content"])
        if match:
            year, month, day = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # Try div.read__time (format: "Kompas.com, 11 Maret 2026")
    read_time = soup.find("div", class_="read__time")
    if read_time:
        text = read_time.get_text(strip=True)
        months = {
            "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
            "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
        }
        match = re.search(r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+(\d{4})', text)
        if match:
            day, month_name, year = match.groups()
            if month_name in months:
                month = months[month_name]
                return f"{year}-{month:02}-{int(day):02}", int(year)

    # Try div.article__date (format: "26/03/2026")
    article_date = soup.find("div", class_="article__date")
    if article_date:
        text = article_date.get_text(strip=True)
        match = re.search(r'(\d{2})/(\d{2})/(\d{4})', text)
        if match:
            day, month, year = match.groups()
            return f"{year}-{month}-{day}", int(year)

    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if not content:
            content = soup.find("article")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE
# ================================================================

all_data = []
year_stats = Counter()

for keyword in KEYWORDS:
    print(f"\n🔍 Mencari: {keyword}")
    encoded = quote_plus(keyword)

    for page in range(1, MAX_PAGES + 1):
        url = f"https://search.kompas.com/search?q={encoded}&page={page}"

        print(f"  Mengambil halaman {page}...")

        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            soup = BeautifulSoup(resp.text, "html.parser")

            links = soup.find_all("a", href=lambda x: x and "/read/" in x)

            if not links:
                print(f"    Tidak ada link artikel di halaman {page}, berhenti.")
                break

            print(f"    Halaman {page}: {len(links)} artikel")

            page_has_data = False

            for link in links:
                try:
                    title = link.text.strip()
                    article_url = link["href"]

                    time.sleep(random.uniform(0.3, 0.5))
                    detail = requests.get(article_url, headers=HEADERS, timeout=10)
                    soup_detail = BeautifulSoup(detail.text, "html.parser")

                    date_str, year = extract_date_from_article(soup_detail)

                    if year:
                        year_stats[year] += 1
                        page_has_data = True

                        # Only save if you want to collect all articles
                        # Uncomment below to save all articles
                        """
                        content = get_content(soup_detail)
                        if content and date_str:
                            all_data.append({
                                "judul": title,
                                "tanggal": date_str,
                                "tahun": year,
                                "url": article_url,
                                "isi": content[:500]
                            })
                        """

                except Exception as e:
                    continue

            # Jika halaman tidak memiliki artikel dengan tanggal yang valid, berhenti
            if not page_has_data:
                print(f"    Tidak ada data valid di halaman {page}, mungkin sudah sampai halaman terakhir.")
                break

        except Exception as e:
            print(f"  Error page {page}: {e}")
            break

    time.sleep(2)

# ================================================================
# HASIL
# ================================================================

print("\n" + "="*60)
print("STATISTIK TAHUN ARTIKEL:")
print("="*60)

if year_stats:
    for year in sorted(year_stats.keys(), reverse=True):
        print(f"  {year}: {year_stats[year]} artikel")

    print("\n" + "="*60)
    print(f"Total artikel unik: {sum(year_stats.values())}")

    # Tampilkan rekomendasi
    print("\n💡 Untuk mengambil artikel tahun tertentu:")
    for year in sorted(year_stats.keys(), reverse=True):
        print(f"   - Tahun {year}: {year_stats[year]} artikel tersedia")

else:
    print("  Tidak ada tanggal yang berhasil diekstrak!")

if all_data:
    df = pd.DataFrame(all_data)
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"\n✅ {len(df)} artikel disimpan ke {OUTPUT_FILE}")


🔍 Mencari: perlindungan data pribadi
  Mengambil halaman 1...
    Halaman 1: 20 artikel
  Mengambil halaman 2...
    Halaman 2: 21 artikel
  Mengambil halaman 3...
    Halaman 3: 19 artikel
  Mengambil halaman 4...
    Halaman 4: 21 artikel
  Mengambil halaman 5...
    Halaman 5: 21 artikel
  Mengambil halaman 6...
    Halaman 6: 20 artikel
  Mengambil halaman 7...
    Halaman 7: 20 artikel
  Mengambil halaman 8...
    Halaman 8: 13 artikel
  Mengambil halaman 9...
    Halaman 9: 1 artikel
  Mengambil halaman 10...
    Halaman 10: 1 artikel
  Mengambil halaman 11...
    Halaman 11: 1 artikel
  Mengambil halaman 12...
    Halaman 12: 1 artikel
  Mengambil halaman 13...
    Halaman 13: 1 artikel
  Mengambil halaman 14...
    Halaman 14: 1 artikel
  Mengambil halaman 15...
    Halaman 15: 1 artikel
  Mengambil halaman 16...
    Halaman 16: 1 artikel
  Mengambil halaman 17...
    Halaman 17: 1 artikel
  Mengambil halaman 18...
    Halaman 18: 1 artikel
  Mengambil halaman 19...
    Halama

In [ ]:
# ============================================================
# scraper_kompas_by_year.py
# Scrape Kompas dengan filter tahun yang akurat
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re
from datetime import datetime

# ================================================================
# KONFIGURASI
# ================================================================
# Tentukan tahun yang ingin di-scrape
TARGET_YEARS = [2020]  # Bisa diisi sesuai kebutuhan

OUTPUT_FILE = f"kompas_pdp_{min(TARGET_YEARS)}_{max(TARGET_YEARS)}.csv"
MAX_PAGES_PER_YEAR = 5  # Maksimal halaman per tahun

KEYWORDS = [
    "perlindungan data pribadi",
    "UU PDP",
    "RUU PDP",
    "kebocoran data",
    "data pribadi bocor",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI EKSTRAK TANGGAL
# ================================================================

def extract_date_from_article(soup):
    """Extract date from article - multiple formats"""

    # Method 1: Meta tag
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date and meta_date.get("content"):
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', meta_date["content"])
        if match:
            year, month, day = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # Method 2: div.read__time (format: "Kompas.com, 11 Maret 2026")
    read_time = soup.find("div", class_="read__time")
    if read_time:
        text = read_time.get_text(strip=True)
        months = {
            "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
            "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
        }
        match = re.search(r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+(\d{4})', text)
        if match:
            day, month_name, year = match.groups()
            if month_name in months:
                month = months[month_name]
                return f"{year}-{month:02}-{int(day):02}", int(year)

    # Method 3: div.article__date (format: "26/03/2026")
    article_date = soup.find("div", class_="article__date")
    if article_date:
        text = article_date.get_text(strip=True)
        match = re.search(r'(\d{2})/(\d{2})/(\d{4})', text)
        if match:
            day, month, year = match.groups()
            return f"{year}-{month}-{day}", int(year)

    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if not content:
            content = soup.find("article")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE PER TAHUN
# ================================================================

all_data = []
total_articles = 0

for year in TARGET_YEARS:
    print(f"\n{'='*60}")
    print(f"📅 Mencari artikel tahun {year}")
    print('='*60)

    # Set filter tanggal untuk tahun tersebut
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    year_data = []

    for keyword in KEYWORDS:
        print(f"\n  🔍 Keyword: {keyword}")
        encoded = quote_plus(keyword)

        for page in range(1, MAX_PAGES_PER_YEAR + 1):
            # URL dengan filter tahun
            url = f"https://search.kompas.com/search?q={encoded}&site_id=all&start_date={start_date}&end_date={end_date}&page={page}"

            try:
                resp = requests.get(url, headers=HEADERS, timeout=10)
                soup = BeautifulSoup(resp.text, "html.parser")

                # Cari link artikel
                links = soup.find_all("a", href=lambda x: x and "/read/" in x)

                if not links:
                    print(f"    Halaman {page}: tidak ada artikel, berhenti")
                    break

                print(f"    Halaman {page}: {len(links)} artikel")

                for link in links:
                    try:
                        title = link.text.strip()
                        article_url = link["href"]

                        # Jeda untuk menghindari blocking
                        time.sleep(random.uniform(0.3, 0.6))

                        # Ambil detail artikel
                        detail = requests.get(article_url, headers=HEADERS, timeout=10)
                        soup_detail = BeautifulSoup(detail.text, "html.parser")

                        # Ekstrak tanggal (untuk verifikasi)
                        date_str, article_year = extract_date_from_article(soup_detail)

                        # Verifikasi tahun (seharusnya sudah sesuai filter, tapi tetap cek)
                        if article_year != year:
                            continue

                        # Ambil konten
                        content = get_content(soup_detail)

                        if content and date_str:
                            article_data = {
                                "tahun": year,
                                "keyword": keyword,
                                "judul": title,
                                "tanggal": date_str,
                                "url": article_url,
                                "isi": content[:1000]  # Simpan 1000 karakter pertama
                            }
                            year_data.append(article_data)
                            print(f"      ✓ {title[:60]} ({date_str})")

                    except Exception as e:
                        continue

                # Jeda antar halaman
                time.sleep(random.uniform(1, 2))

            except Exception as e:
                print(f"    Error halaman {page}: {e}")
                break

    # Simpan data per tahun
    if year_data:
        # Hapus duplikat berdasarkan URL
        unique_data = {item['url']: item for item in year_data}.values()
        year_unique = list(unique_data)

        all_data.extend(year_unique)
        print(f"\n  ✅ Tahun {year}: {len(year_unique)} artikel unik")
        total_articles += len(year_unique)
    else:
        print(f"\n  ⚠️ Tahun {year}: tidak ada artikel ditemukan")

    # Jeda antar tahun
    time.sleep(random.uniform(2, 3))

# ================================================================
# SAVE RESULTS
# ================================================================

print(f"\n{'='*60}")
print("HASIL SCRAPING")
print('='*60)

if all_data:
    df = pd.DataFrame(all_data)

    # Urutkan berdasarkan tahun dan tanggal
    df = df.sort_values(['tahun', 'tanggal'], ascending=[False, False])

    # Simpan ke CSV
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n✅ Total artikel: {len(df)}")
    print(f"✅ Disimpan ke: {OUTPUT_FILE}")

    # Statistik per tahun
    print("\n📊 Statistik per tahun:")
    year_stats = df['tahun'].value_counts().sort_index(ascending=False)
    for year, count in year_stats.items():
        print(f"   {year}: {count} artikel")

    # Statistik per keyword
    print("\n📊 Statistik per keyword:")
    keyword_stats = df['keyword'].value_counts()
    for keyword, count in keyword_stats.items():
        print(f"   {keyword}: {count} artikel")

    # Sample data
    print("\n📄 Sample artikel (5 terbaru):")
    print(df[['tahun', 'tanggal', 'judul', 'keyword']].head())

else:
    print("\n❌ Tidak ada data yang berhasil di-scrape")

print(f"\n{'='*60}")
print("SELESAI!")
print('='*60)


📅 Mencari artikel tahun 2020

  🔍 Keyword: perlindungan data pribadi
    Halaman 1: 21 artikel
      ✓ Pemerintah Didesak Sahkan RUU Perlindungan Data Pribadi




 (2020-05-06)
      ✓ RUU Perlindungan Data Pribadi Ditargetkan Selesai Oktober 20 (2020-07-28)
      ✓ Soal RUU Perlindungan Data Pribadi, "Bola" Kini Ada di DPR

 (2020-01-28)
      ✓ Menkoinfo Berharap DPR Segera Bahas Draft RUU Perlindungan D (2020-01-28)
      ✓ RUU Perlindungan Data Pribadi Segera Dibahas DPR dan Pemerin (2020-02-05)
      ✓ Wabah Covid-19, Menkominfo Berkomitmen Jaga Perlindungan Dat (2020-04-29)
      ✓ Ini 12 Poin yang Diatur dalam RUU Perlindungan Data Pribadi
 (2020-02-25)
      ✓ Pemerintah Serahkan Draft RUU Perlindungan Data Pribadi ke D (2020-01-28)
      ✓ Rapat, DPR-Kominfo Bahas DIM RUU Perlindungan Data Pribadi

 (2020-11-30)
      ✓ IDA Gandeng ABDI Bahas Pentingnya Regulasi Perlindungan Data (2020-11-05)
      ✓ Sanksi Pidana di RUU Perlindungan Data Pribadi Diminta Dihap (2020-07-09)
  

KeyboardInterrupt: 

In [ ]:
# ============================================================
# scraper_kompas_by_year.py
# Scrape Kompas dengan filter tahun yang akurat
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re
from datetime import datetime

# ================================================================
# KONFIGURASI
# ================================================================
# Tentukan tahun yang ingin di-scrape
TARGET_YEARS = [2020]  # Bisa diisi sesuai kebutuhan

OUTPUT_FILE = f"kompas_pdp_{min(TARGET_YEARS)}_{max(TARGET_YEARS)}.csv"
MAX_PAGES_PER_YEAR = 2  # Maksimal halaman per tahun

KEYWORDS = [
# ===============================
    # Regulasi & Hukum
    # ===============================
    "perlindungan data pribadi",
    # "UU Perlindungan Data Pribadi",
    # "RUU PDP",
    # "undang-undang perlindungan data",
    # "regulasi data pribadi",
    # "hukum data pribadi",
    # "aturan data pribadi",
    # "kebijakan data pribadi",

    # # ===============================
    # # Kebocoran & Serangan Siber
    # # ===============================
    # "kasus kebocoran data",
    # "kebocoran data",
    # "data pribadi bocor",
    # "data bocor",
    # "serangan siber data",
    # "peretasan data",
    # "serangan siber Indonesia",

    # # ===============================
    # # Lembaga & Aktor
    # # ===============================
    # "Kominfo data pribadi",
    # "Kominfo kebocoran data",
    # "Kemenkominfo data bocor",

    # # ===============================
    # # Kasus Spesifik Indonesia
    # # ===============================
    # "kebocoran data BPJS",
    # "kebocoran data e-KTP",
    # "kebocoran data Dukcapil",
    # "ransomware Indonesia",
    # "data bocor Tokopedia",
    # "data bocor Bukalapak",
    # "data bocor IndiHome",

    # # ===============================
    # # Tambahan Penelitian
    # # ===============================
    # "keamanan data pribadi",
    # "privasi data Indonesia",
    # "perlindungan privasi",
    # "keamanan siber Indonesia",
    # "insiden kebocoran data Indonesia",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI EKSTRAK TANGGAL
# ================================================================

def extract_date_from_article(soup):
    """Extract date from article - multiple formats"""

    # Method 1: Meta tag
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date and meta_date.get("content"):
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', meta_date["content"])
        if match:
            year, month, day = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # Method 2: div.read__time (format: "Kompas.com, 11 Maret 2026")
    read_time = soup.find("div", class_="read__time")
    if read_time:
        text = read_time.get_text(strip=True)
        months = {
            "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
            "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
        }
        match = re.search(r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+(\d{4})', text)
        if match:
            day, month_name, year = match.groups()
            if month_name in months:
                month = months[month_name]
                return f"{year}-{month:02}-{int(day):02}", int(year)

    # Method 3: div.article__date (format: "26/03/2026")
    article_date = soup.find("div", class_="article__date")
    if article_date:
        text = article_date.get_text(strip=True)
        match = re.search(r'(\d{2})/(\d{2})/(\d{4})', text)
        if match:
            day, month, year = match.groups()
            return f"{year}-{month}-{day}", int(year)

    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if not content:
            content = soup.find("article")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE PER TAHUN
# ================================================================

all_data = []
total_articles = 0

for year in TARGET_YEARS:
    print(f"\n{'='*60}")
    print(f"📅 Mencari artikel tahun {year}")
    print('='*60)

    # Set filter tanggal untuk tahun tersebut
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    year_data = []

    for keyword in KEYWORDS:
        print(f"\n  🔍 Keyword: {keyword}")
        encoded = quote_plus(keyword)

        for page in range(1, MAX_PAGES_PER_YEAR + 1):
            # URL dengan filter tahun
            url = f"https://search.kompas.com/search?q={encoded}&site_id=all&start_date={start_date}&end_date={end_date}&page={page}"

            try:
                resp = requests.get(url, headers=HEADERS, timeout=10)
                soup = BeautifulSoup(resp.text, "html.parser")

                # Cari link artikel
                links = soup.find_all("a", href=lambda x: x and "/read/" in x)

                if not links:
                    print(f"    Halaman {page}: tidak ada artikel, berhenti")
                    break

                print(f"    Halaman {page}: {len(links)} artikel")

                for link in links:
                    try:
                        title = link.text.strip()
                        article_url = link["href"]

                        # Jeda untuk menghindari blocking
                        time.sleep(random.uniform(0.3, 0.6))

                        # Ambil detail artikel
                        detail = requests.get(article_url, headers=HEADERS, timeout=10)
                        soup_detail = BeautifulSoup(detail.text, "html.parser")

                        # Ekstrak tanggal (untuk verifikasi)
                        date_str, article_year = extract_date_from_article(soup_detail)

                        # Verifikasi tahun (seharusnya sudah sesuai filter, tapi tetap cek)
                        if article_year != year:
                            continue

                        # Ambil konten
                        content = get_content(soup_detail)

                        if content and date_str:
                            article_data = {
                                "tahun": year,
                                "keyword": keyword,
                                "judul": title,
                                "tanggal": date_str,
                                "url": article_url,
                                "isi": content[:1000]  # Simpan 1000 karakter pertama
                            }
                            year_data.append(article_data)
                            print(f"      ✓ {title[:60]} ({date_str})")

                    except Exception as e:
                        continue

                # Jeda antar halaman
                time.sleep(random.uniform(1, 2))

            except Exception as e:
                print(f"    Error halaman {page}: {e}")
                break

    # Simpan data per tahun
    if year_data:
        # Hapus duplikat berdasarkan URL
        unique_data = {item['url']: item for item in year_data}.values()
        year_unique = list(unique_data)

        all_data.extend(year_unique)
        print(f"\n  ✅ Tahun {year}: {len(year_unique)} artikel unik")
        total_articles += len(year_unique)
    else:
        print(f"\n  ⚠️ Tahun {year}: tidak ada artikel ditemukan")

    # Jeda antar tahun
    time.sleep(random.uniform(2, 3))

# ================================================================
# SAVE RESULTS
# ================================================================

print(f"\n{'='*60}")
print("HASIL SCRAPING")
print('='*60)

if all_data:
    df = pd.DataFrame(all_data)

    # Urutkan berdasarkan tahun dan tanggal
    df = df.sort_values(['tahun', 'tanggal'], ascending=[False, False])

    # Simpan ke CSV
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n✅ Total artikel: {len(df)}")
    print(f"✅ Disimpan ke: {OUTPUT_FILE}")

    # Statistik per tahun
    print("\n📊 Statistik per tahun:")
    year_stats = df['tahun'].value_counts().sort_index(ascending=False)
    for year, count in year_stats.items():
        print(f"   {year}: {count} artikel")

    # Statistik per keyword
    print("\n📊 Statistik per keyword:")
    keyword_stats = df['keyword'].value_counts()
    for keyword, count in keyword_stats.items():
        print(f"   {keyword}: {count} artikel")

    # Sample data
    print("\n📄 Sample artikel (5 terbaru):")
    print(df[['tahun', 'tanggal', 'judul', 'keyword']].head())

else:
    print("\n❌ Tidak ada data yang berhasil di-scrape")

print(f"\n{'='*60}")
print("SELESAI!")
print('='*60)


📅 Mencari artikel tahun 2020

  🔍 Keyword: perlindungan data pribadi
    Halaman 1: 21 artikel
      ✓ Pemerintah Didesak Sahkan RUU Perlindungan Data Pribadi




 (2020-05-06)
      ✓ RUU Perlindungan Data Pribadi Ditargetkan Selesai Oktober 20 (2020-07-28)
      ✓ Soal RUU Perlindungan Data Pribadi, "Bola" Kini Ada di DPR

 (2020-01-28)
      ✓ Menkoinfo Berharap DPR Segera Bahas Draft RUU Perlindungan D (2020-01-28)
      ✓ RUU Perlindungan Data Pribadi Segera Dibahas DPR dan Pemerin (2020-02-05)
      ✓ Wabah Covid-19, Menkominfo Berkomitmen Jaga Perlindungan Dat (2020-04-29)
      ✓ Ini 12 Poin yang Diatur dalam RUU Perlindungan Data Pribadi
 (2020-02-25)
      ✓ Pemerintah Serahkan Draft RUU Perlindungan Data Pribadi ke D (2020-01-28)
      ✓ Rapat, DPR-Kominfo Bahas DIM RUU Perlindungan Data Pribadi

 (2020-11-30)
      ✓ IDA Gandeng ABDI Bahas Pentingnya Regulasi Perlindungan Data (2020-11-05)
      ✓ Sanksi Pidana di RUU Perlindungan Data Pribadi Diminta Dihap (2020-07-09)
  

In [ ]:
# ============================================================
# scraper_kompas_final.py
# Output format: source, keyword, title, date, year, url, content
# Dengan deduplikasi berdasarkan judul
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re

# ================================================================
# KONFIGURASI
# ================================================================
TARGET_YEARS = [2020]  # Bisa disesuaikan
OUTPUT_FILE = f"kompas_pdp_all_years.csv"
MAX_PAGES_PER_YEAR = 15  # Sesuaikan kebutuhan

# KEYWORD YANG DIPERBARUI
KEYWORDS = [
    # Regulasi & Sentimen Utama
    "UU PDP",
    "RUU PDP",
    "Undang-Undang Perlindungan Data Pribadi",
    "perlindungan data pribadi",

    # Insiden & Krisis
    "kebocoran data",
    "peretasan data",
    "keamanan siber Indonesia",

    # Aktor & Kritik (Penting untuk analisis sentimen)
    "Kominfo data pribadi",

    # Kasus Besar (Sebagai Anchor/Jangkar data)
    "kebocoran data BPJS",
    "kebocoran data Tokopedia",
    "kebocoran data e-KTP"
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI FILTER ARTIKEL
# ================================================================

def is_relevant_article(title, content):
    """Cek apakah artikel relevan dengan topik PDP (bukan COVID)"""

    text = (title + " " + content).lower()

    # Cek apakah mengandung keyword yang diinginkan
    pdp_keywords = ["uu pdp", "ruu pdp", "perlindungan data pribadi", "data pribadi",
                    "kebocoran data", "peretasan data", "keamanan siber"]
    has_pdp_keyword = any(keyword in text for keyword in pdp_keywords)

    if not has_pdp_keyword:
        return False

    # Cek apakah mengandung kata-kata COVID (kecuali jika juga mengandung UU/RUU PDP)
    covid_words = ["covid", "corona", "pandemi", "odp", "pasien dalam pengawasan", "virus corona"]

    for covid_word in covid_words:
        if covid_word in text:
            # Jika mengandung UU PDP atau RUU PDP di dekatnya, tetap dianggap relevan
            if "uu pdp" in text or "ruu pdp" in text:
                continue
            return False

    return True

def extract_date_from_article(soup):
    """Extract date from article"""

    # Method 1: Meta tag
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date and meta_date.get("content"):
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', meta_date["content"])
        if match:
            year, month, day = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # Method 2: div.read__time
    read_time = soup.find("div", class_="read__time")
    if read_time:
        text = read_time.get_text(strip=True)
        months = {
            "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
            "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
        }
        match = re.search(r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+(\d{4})', text)
        if match:
            day, month_name, year = match.groups()
            if month_name in months:
                month = months[month_name]
                return f"{year}-{month:02}-{int(day):02}", int(year)

    # Method 3: div.article__date
    article_date = soup.find("div", class_="article__date")
    if article_date:
        text = article_date.get_text(strip=True)
        match = re.search(r'(\d{2})/(\d{2})/(\d{4})', text)
        if match:
            day, month, year = match.groups()
            return f"{year}-{month}-{day}", int(year)

    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if not content:
            content = soup.find("article")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE
# ================================================================

all_data = []
skipped_covid = 0
total_processed = 0

print("\n" + "="*70)
print("📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI")
print("="*70)
print(f"📅 Tahun: {', '.join(map(str, TARGET_YEARS))}")
print(f"🔍 Keyword: {len(KEYWORDS)} keyword")
print("="*70)

for year in TARGET_YEARS:
    print(f"\n📅 Mengolah tahun: {year}")
    print("-" * 50)

    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    year_data = []

    for keyword in KEYWORDS:
        print(f"\n  🔍 Keyword: {keyword}")
        encoded = quote_plus(keyword)

        for page in range(1, MAX_PAGES_PER_YEAR + 1):
            url = f"https://search.kompas.com/search?q={encoded}&site_id=all&start_date={start_date}&end_date={end_date}&page={page}"

            try:
                resp = requests.get(url, headers=HEADERS, timeout=10)
                soup = BeautifulSoup(resp.text, "html.parser")

                links = soup.find_all("a", href=lambda x: x and "/read/" in x)

                if not links:
                    break

                print(f"    Halaman {page}: {len(links)} artikel", end="")

                page_articles = 0
                for link in links:
                    try:
                        title = link.text.strip()
                        article_url = link["href"]

                        time.sleep(random.uniform(0.3, 0.5))

                        detail = requests.get(article_url, headers=HEADERS, timeout=10)
                        soup_detail = BeautifulSoup(detail.text, "html.parser")

                        date_str, article_year = extract_date_from_article(soup_detail)

                        if article_year != year:
                            continue

                        content = get_content(soup_detail)

                        if not content:
                            continue

                        # Filter artikel relevan
                        if is_relevant_article(title, content):
                            article_data = {
                                "source": "Kompas",
                                "keyword": keyword,
                                "title": title,
                                "date": date_str,
                                "year": article_year,
                                "url": article_url,
                                "content": content
                            }
                            year_data.append(article_data)
                            page_articles += 1
                            total_processed += 1
                        else:
                            if "odp" in title.lower() or "pdp" in title.lower():
                                skipped_covid += 1

                    except Exception as e:
                        continue

                print(f" → {page_articles} relevan")

                time.sleep(random.uniform(1, 2))

                if page_articles == 0 and page > 1:
                    break

            except Exception as e:
                print(f"    Error: {e}")
                break

    if year_data:
        # Hapus duplikat berdasarkan URL dalam satu tahun
        unique_by_url = {}
        for item in year_data:
            if item['url'] not in unique_by_url:
                unique_by_url[item['url']] = item

        year_unique = list(unique_by_url.values())
        all_data.extend(year_unique)
        print(f"\n  ✅ Tahun {year}: {len(year_unique)} artikel unik (dari {len(year_data)} total)")
    else:
        print(f"\n  ⚠️ Tahun {year}: tidak ada artikel relevan")

# ================================================================
# DEDUPLIKASI BERDASARKAN JUDUL (TAMBAHAN)
# ================================================================

print(f"\n{'='*70}")
print("📊 PROSES DEDUPLIKASI DATA")
print('='*70)

if all_data:
    df = pd.DataFrame(all_data)

    print(f"Total data sebelum deduplikasi: {len(df)}")

    # Deduplikasi berdasarkan judul (case insensitive)
    df['title_lower'] = df['title'].str.lower().str.strip()
    df_unique_title = df.drop_duplicates(subset=['title_lower'], keep='first')

    # Jumlah duplikat yang dihapus
    duplicate_count = len(df) - len(df_unique_title)
    print(f"Duplikat berdasarkan judul: {duplicate_count} artikel dihapus")

    # Hapus kolom temporary
    df_final = df_unique_title.drop(columns=['title_lower'])

    # Deduplikasi tambahan berdasarkan URL (jika masih ada)
    df_final = df_final.drop_duplicates(subset=['url'], keep='first')
    print(f"Duplikat berdasarkan URL: {len(df_unique_title) - len(df_final)} artikel dihapus")

    print(f"Total data setelah deduplikasi: {len(df_final)}")

    # Urutkan berdasarkan tahun dan tanggal
    df_final = df_final.sort_values(['year', 'date'], ascending=[False, False])

    # Simpan ke CSV
    df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n✅ Disimpan ke: {OUTPUT_FILE}")

    # ================================================================
    # STATISTIK LENGKAP
    # ================================================================

    print(f"\n{'='*70}")
    print("STATISTIK DATA")
    print('='*70)

    print(f"\n📊 Statistik per tahun:")
    year_stats = df_final['year'].value_counts().sort_index(ascending=False)
    for year, count in year_stats.items():
        print(f"   {year}: {count} artikel")

    print(f"\n📊 Statistik per keyword:")
    keyword_stats = df_final['keyword'].value_counts()
    for keyword, count in keyword_stats.items():
        print(f"   {keyword}: {count} artikel")

    print(f"\n📊 Statistik COVID yang dilewati:")
    print(f"   Artikel COVID yang difilter: {skipped_covid}")

    print(f"\n📄 Sample data (10 artikel terbaru):")
    print(df_final[['source', 'title', 'date', 'year', 'keyword']].head(10).to_string())

    # Contoh format data
    print(f"\n📋 Contoh format data:")
    if len(df_final) > 0:
        print(f"   source: {df_final['source'].iloc[0]}")
        print(f"   keyword: {df_final['keyword'].iloc[0]}")
        print(f"   title: {df_final['title'].iloc[0][:80]}...")
        print(f"   date: {df_final['date'].iloc[0]}")
        print(f"   year: {df_final['year'].iloc[0]}")
        print(f"   url: {df_final['url'].iloc[0]}")
        print(f"   content: {df_final['content'].iloc[0][:150]}...")

    # Informasi file
    print(f"\n📁 File: {OUTPUT_FILE}")
    print(f"📊 Total baris: {len(df_final)}")
    print(f"📊 Total kolom: {len(df_final.columns)}")
    print(f"📊 Kolom: {', '.join(df_final.columns)}")

    # Simpan juga statistik ke file terpisah (opsional)
    stats_file = OUTPUT_FILE.replace('.csv', '_stats.txt')
    with open(stats_file, 'w', encoding='utf-8') as f:
        f.write("="*70 + "\n")
        f.write("STATISTIK SCRAPING KOMPAS\n")
        f.write("="*70 + "\n\n")
        f.write(f"Total artikel: {len(df_final)}\n")
        f.write(f"Tahun: {', '.join(map(str, TARGET_YEARS))}\n\n")
        f.write("Distribusi per tahun:\n")
        for year, count in year_stats.items():
            f.write(f"  {year}: {count}\n")
        f.write("\nDistribusi per keyword:\n")
        for keyword, count in keyword_stats.items():
            f.write(f"  {keyword}: {count}\n")
        f.write(f"\nArtikel COVID yang difilter: {skipped_covid}\n")

    print(f"Statistik disimpan ke: {stats_file}")

else:
    print("\nTidak ada artikel relevan yang ditemukan")

print(f"\n{'='*70}")
print("✅ SELESAI!")
print('='*70)


📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI
📅 Tahun: 2020
🔍 Keyword: 11 keyword

📅 Mengolah tahun: 2020
--------------------------------------------------

  🔍 Keyword: UU PDP
    Halaman 1: 21 artikel → 5 relevan
    Halaman 2: 21 artikel → 0 relevan

  🔍 Keyword: RUU PDP
    Halaman 1: 21 artikel → 20 relevan
    Halaman 2: 21 artikel

KeyboardInterrupt: 

In [ ]:
# ============================================================
# scraper_kompas_final.py
# Output format: source, keyword, title, date, year, url, content
# Dengan deduplikasi berdasarkan judul
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re

# ================================================================
# KONFIGURASI
# ================================================================
TARGET_YEARS = [2020]  # Bisa disesuaikan
OUTPUT_FILE = f"kompas_pdp2020_1p_test.csv"
MAX_PAGES_PER_YEAR = 1  # Sesuaikan kebutuhan

# KEYWORD
KEYWORDS = [
    # Regulasi & Sentimen Utama
    "UU PDP",
    "RUU PDP",
    "Undang-Undang Perlindungan Data Pribadi",
    "perlindungan data pribadi",

    # Insiden & Krisis
    "kebocoran data",
    "peretasan data",
    "keamanan siber Indonesia",

    # Aktor & Kritik (Penting untuk analisis sentimen)
    "Kominfo data pribadi",

    # Kasus Besar (Sebagai Anchor/Jangkar data)
    "kebocoran data BPJS",
    "kebocoran data Tokopedia",
    "kebocoran data e-KTP"
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI EKSTRAK TANGGAL
# ================================================================

def extract_date_from_article(soup):
    """Extract date from article"""

    # Method 1: Meta tag
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date and meta_date.get("content"):
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', meta_date["content"])
        if match:
            year, month, day = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # Method 2: div.read__time
    read_time = soup.find("div", class_="read__time")
    if read_time:
        text = read_time.get_text(strip=True)
        months = {
            "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
            "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
        }
        match = re.search(r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+(\d{4})', text)
        if match:
            day, month_name, year = match.groups()
            if month_name in months:
                month = months[month_name]
                return f"{year}-{month:02}-{int(day):02}", int(year)

    # Method 3: div.article__date
    article_date = soup.find("div", class_="article__date")
    if article_date:
        text = article_date.get_text(strip=True)
        match = re.search(r'(\d{2})/(\d{2})/(\d{4})', text)
        if match:
            day, month, year = match.groups()
            return f"{year}-{month}-{day}", int(year)

    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if not content:
            content = soup.find("article")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE
# ================================================================

all_data = []
total_processed = 0
total_articles_found = 0

print("\n" + "="*80)
print("📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI")
print("="*80)
print(f"📅 Tahun: {', '.join(map(str, TARGET_YEARS))}")
print(f"🔍 Keyword: {len(KEYWORDS)} keyword")
print("="*80)

for year in TARGET_YEARS:
    print(f"\n{'='*60}")
    print(f"📅 TAHUN {year}")
    print('='*60)

    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    year_data = []
    year_articles_count = 0

    for keyword in KEYWORDS:
        print(f"\n  🔍 Keyword: {keyword}")
        encoded = quote_plus(keyword)

        for page in range(1, MAX_PAGES_PER_YEAR + 1):
            url = f"https://search.kompas.com/search?q={encoded}&site_id=all&start_date={start_date}&end_date={end_date}&page={page}"

            try:
                resp = requests.get(url, headers=HEADERS, timeout=10)
                soup = BeautifulSoup(resp.text, "html.parser")

                links = soup.find_all("a", href=lambda x: x and "/read/" in x)

                if not links:
                    break

                print(f"    Halaman {page}: {len(links)} artikel ditemukan")

                page_articles = 0
                for link in links:
                    try:
                        title = link.text.strip()
                        article_url = link["href"]

                        time.sleep(random.uniform(0.3, 0.5))

                        detail = requests.get(article_url, headers=HEADERS, timeout=10)
                        soup_detail = BeautifulSoup(detail.text, "html.parser")

                        date_str, article_year = extract_date_from_article(soup_detail)

                        if article_year != year:
                            continue

                        content = get_content(soup_detail)

                        if not content:
                            continue

                        article_data = {
                            "source": "Kompas",
                            "keyword": keyword,
                            "title": title,
                            "date": date_str,
                            "year": article_year,
                            "url": article_url,
                            "content": content
                        }
                        year_data.append(article_data)
                        page_articles += 1
                        total_processed += 1
                        total_articles_found += 1

                        # TAMPILKAN JUDUL BERITA YANG BERHASIL DIAMBIL
                        print(f"      ✓ {title[:80]} → {date_str}")

                    except Exception as e:
                        continue

                print(f"    📊 Halaman {page}: {page_articles} artikel berhasil diambil")

                time.sleep(random.uniform(1, 2))

                if page_articles == 0 and page > 1:
                    break

            except Exception as e:
                print(f"    ❌ Error: {e}")
                break

    if year_data:
        # Hapus duplikat berdasarkan URL dalam satu tahun
        unique_by_url = {}
        for item in year_data:
            if item['url'] not in unique_by_url:
                unique_by_url[item['url']] = item

        year_unique = list(unique_by_url.values())
        all_data.extend(year_unique)
        print(f"\n  ✅ TAHUN {year}: {len(year_unique)} artikel unik (dari {len(year_data)} total)")
        print(f"  📝 Contoh judul di tahun {year}:")
        for i, item in enumerate(year_unique[:5]):
            print(f"     {i+1}. {item['title'][:80]}")
    else:
        print(f"\n  ⚠️ TAHUN {year}: tidak ada artikel ditemukan")

# ================================================================
# DEDUPLIKASI BERDASARKAN JUDUL
# ================================================================

print(f"\n{'='*80}")
print("📊 PROSES DEDUPLIKASI DATA")
print('='*80)

if all_data:
    df = pd.DataFrame(all_data)

    print(f"📊 Total data sebelum deduplikasi: {len(df)} artikel")

    # Deduplikasi berdasarkan judul (case insensitive)
    df['title_lower'] = df['title'].str.lower().str.strip()
    df_unique_title = df.drop_duplicates(subset=['title_lower'], keep='first')

    duplicate_by_title = len(df) - len(df_unique_title)
    print(f"📊 Duplikat berdasarkan judul: {duplicate_by_title} artikel dihapus")

    # Hapus kolom temporary
    df_final = df_unique_title.drop(columns=['title_lower'])

    # Deduplikasi tambahan berdasarkan URL
    df_final = df_final.drop_duplicates(subset=['url'], keep='first')
    duplicate_by_url = len(df_unique_title) - len(df_final)
    print(f"📊 Duplikat berdasarkan URL: {duplicate_by_url} artikel dihapus")

    print(f"📊 Total data setelah deduplikasi: {len(df_final)} artikel")

    # Urutkan berdasarkan tahun dan tanggal
    df_final = df_final.sort_values(['year', 'date'], ascending=[False, False])

    # Simpan ke CSV
    df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n✅ File disimpan ke: {OUTPUT_FILE}")

    # ================================================================
    # STATISTIK LENGKAP
    # ================================================================

    print(f"\n{'='*80}")
    print("📊 STATISTIK DATA")
    print('='*80)

    print(f"\n📊 Statistik per tahun:")
    year_stats = df_final['year'].value_counts().sort_index(ascending=False)
    for year, count in year_stats.items():
        print(f"   {year}: {count} artikel")

    print(f"\n📊 Statistik per keyword:")
    keyword_stats = df_final['keyword'].value_counts()
    for keyword, count in keyword_stats.items():
        print(f"   {keyword}: {count} artikel")

    # TAMPILKAN JUDUL BERITA TERBARU
    print(f"\n📰 10 ARTIKEL TERBARU (setelah deduplikasi):")
    print("-" * 80)
    for i, row in df_final.head(10).iterrows():
        print(f"{i+1:2d}. [{row['year']}] {row['date']} | {row['keyword']}")
        print(f"    📌 {row['title'][:100]}")
        print(f"    🔗 {row['url'][:80]}")
        print()

    # TAMPILKAN SAMPLE JUDUL PER TAHUN
    print(f"\n📰 SAMPLE JUDUL PER TAHUN:")
    print("-" * 80)
    for year in sorted(df_final['year'].unique(), reverse=True):
        year_df = df_final[df_final['year'] == year]
        print(f"\n📅 TAHUN {year} ({len(year_df)} artikel):")
        for i, row in year_df.head(3).iterrows():
            print(f"   • {row['title'][:90]}")
        if len(year_df) > 3:
            print(f"   ... dan {len(year_df) - 3} artikel lainnya")

    # Informasi file
    print(f"\n{'='*80}")
    print("📁 INFORMASI FILE")
    print('='*80)
    print(f"📄 Nama file: {OUTPUT_FILE}")
    print(f"📊 Jumlah artikel: {len(df_final)}")
    print(f"📊 Jumlah kolom: {len(df_final.columns)}")
    print(f"📊 Kolom: {', '.join(df_final.columns)}")
    print(f"📊 Rentang tahun: {df_final['year'].min()} - {df_final['year'].max()}")

    # Simpan ringkasan ke file
    summary_file = OUTPUT_FILE.replace('.csv', '_summary.txt')
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("RINGKASAN SCRAPING KOMPAS\n")
        f.write("="*80 + "\n\n")
        f.write(f"Total artikel: {len(df_final)}\n")
        f.write(f"Total artikel diproses: {total_processed}\n")
        f.write(f"Total artikel ditemukan: {total_articles_found}\n\n")

        f.write("Distribusi per tahun:\n")
        for year, count in year_stats.items():
            f.write(f"  {year}: {count}\n")

        f.write("\nDistribusi per keyword:\n")
        for keyword, count in keyword_stats.items():
            f.write(f"  {keyword}: {count}\n")

        f.write("\n" + "="*80 + "\n")
        f.write("10 ARTIKEL TERBARU\n")
        f.write("="*80 + "\n")
        for i, row in df_final.head(10).iterrows():
            f.write(f"\n{i+1}. [{row['year']}] {row['date']}\n")
            f.write(f"   Keyword: {row['keyword']}\n")
            f.write(f"   Judul: {row['title']}\n")
            f.write(f"   URL: {row['url']}\n")

    print(f"\n📊 Ringkasan disimpan ke: {summary_file}")

else:
    print("\n❌ Tidak ada artikel yang berhasil di-scrape")
    print("\n💡 Saran:")
    print("   1. Coba tambah MAX_PAGES_PER_YEAR (misal 30-50)")
    print("   2. Periksa koneksi internet")
    print("   3. Coba tahun lain seperti 2021, 2022, 2023")

print(f"\n{'='*80}")
print("✅ PROSES SELESAI!")
print('='*80)
print(f"📊 Total artikel yang berhasil diambil: {len(df_final) if all_data else 0}")
print(f"📁 File output: {OUTPUT_FILE if all_data else 'Tidak ada data'}")
print('='*80)


📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI
📅 Tahun: 2020
🔍 Keyword: 11 keyword

📅 TAHUN 2020

  🔍 Keyword: UU PDP
    Halaman 1: 21 artikel ditemukan
      ✓ ELSAM: Harus Ada Pengawas UU PDP di Luar Pemerintah




                         → 2020-01-31
      ✓ Sanksi soal Data Pribadi yang Sudah Diatur di UU Lain Dinilai Tak Perlu Masuk RU → 2020-08-10
      ✓ Asosiasi Telekomunikasi Ingin Ada Pengawas Independen UU PDP di Luar Pemerintah
 → 2020-07-09
      ✓ Serial Infografik Virus Corona: Apa Itu PDP?




                                → 2020-03-25
      ✓ Pembahasan RUU PDP Ditargetkan Rampung November 2020




                        → 2020-09-01
      ✓ Isolasi Mandiri, PDP di Salatiga Meninggal Dunia




                            → 2020-04-24
      ✓ Tiga PDP Corona di Lampung Dinyatakan Negatif




                               → 2020-03-22
      ✓ Warga Kapuas Tolak Pemakaman Jenazah PDP




                                    → 2020-05-14
      ✓ 1 PDP Meninggal Dunia 

**FIX FINAL CORRECT**

In [ ]:
# ============================================================
# scraper_kompas_final.py
# Dengan perbaikan pengambilan judul yang lebih akurat
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re

# ================================================================
# KONFIGURASI
# ================================================================
TARGET_YEARS = [2020]
OUTPUT_FILE = f"kompas_pdp_2020_new.csv"
MAX_PAGES_PER_YEAR = 1

KEYWORDS = [
    "Perlindungan Data Pribadi",
    "UU Perlindungan Data Pribadi",
    "RUU Perlindungan Data Pribadi",
    "Undang-Undang Perlindungan Data Pribadi",
    "perlindungan data pribadi",
    "kebocoran data pribadi",
    "Kominfo data pribadi",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI EKSTRAK JUDUL YANG LEBIH AKURAT
# ================================================================

def extract_title_from_search_result(article_element):
    """Extract clean title from search result article element"""

    # Method 1: Cari h3 atau h2 yang berisi judul
    title_tag = article_element.find(['h3', 'h2'])
    if title_tag:
        # Ambil teks dari h3/h2, bersihkan dari newline dan spasi berlebih
        title = title_tag.get_text(strip=True)
        if title:
            return title

    # Method 2: Cari link dengan class tertentu
    link = article_element.find('a', class_=re.compile(r'title|article'))
    if link:
        title = link.get_text(strip=True)
        if title:
            return title

    # Method 3: Cari elemen dengan class yang mengandung kata 'title'
    title_elem = article_element.find(class_=re.compile(r'title', re.I))
    if title_elem:
        title = title_elem.get_text(strip=True)
        if title:
            return title

    # Method 4: Ambil dari link pertama yang mengandung /read/
    link = article_element.find('a', href=lambda x: x and '/read/' in x)
    if link:
        # Ambil teks, tapi pisahkan dengan baris baru
        full_text = link.get_text(separator='\n', strip=True)
        # Ambil baris pertama (biasanya judul)
        lines = [line.strip() for line in full_text.split('\n') if line.strip()]
        if lines:
            return lines[0]

    return None

def extract_date_from_article(soup):
    """Extract date from article"""

    # Method 1: Meta tag
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date and meta_date.get("content"):
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', meta_date["content"])
        if match:
            year, month, day = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # Method 2: div.read__time
    read_time = soup.find("div", class_="read__time")
    if read_time:
        text = read_time.get_text(strip=True)
        months = {
            "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
            "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
        }
        match = re.search(r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+(\d{4})', text)
        if match:
            day, month_name, year = match.groups()
            if month_name in months:
                month = months[month_name]
                return f"{year}-{month:02}-{int(day):02}", int(year)

    # Method 3: div.article__date
    article_date = soup.find("div", class_="article__date")
    if article_date:
        text = article_date.get_text(strip=True)
        match = re.search(r'(\d{2})/(\d{2})/(\d{4})', text)
        if match:
            day, month, year = match.groups()
            return f"{year}-{month}-{day}", int(year)

    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if not content:
            content = soup.find("article")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE
# ================================================================

all_data = []
total_processed = 0

print("\n" + "="*80)
print("📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI")
print("="*80)
print(f"📅 Tahun: {', '.join(map(str, TARGET_YEARS))}")
print(f"🔍 Keyword: {len(KEYWORDS)} keyword")
print("="*80)

for year in TARGET_YEARS:
    print(f"\n{'='*60}")
    print(f"📅 TAHUN {year}")
    print('='*60)

    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    year_data = []

    for keyword in KEYWORDS:
        print(f"\n  🔍 Keyword: {keyword}")
        encoded = quote_plus(keyword)

        for page in range(1, MAX_PAGES_PER_YEAR + 1):
            url = f"https://search.kompas.com/search?q={encoded}&site_id=all&start_date={start_date}&end_date={end_date}&page={page}"

            try:
                resp = requests.get(url, headers=HEADERS, timeout=10)
                soup = BeautifulSoup(resp.text, "html.parser")

                # Cari elemen artikel
                article_elements = soup.find_all("div", class_=re.compile(r'article', re.I))

                # Filter yang memiliki link /read/
                valid_articles = []
                for article in article_elements:
                    links = article.find_all("a", href=lambda x: x and "/read/" in x)
                    if links:
                        valid_articles.append(article)

                if not valid_articles:
                    # Coba cari link langsung
                    links = soup.find_all("a", href=lambda x: x and "/read/" in x)
                    if not links:
                        break
                    # Buat wrapper untuk setiap link
                    valid_articles = [link.parent for link in links[:10]]

                print(f"    Halaman {page}: {len(valid_articles)} artikel ditemukan")

                page_articles = 0
                for article_elem in valid_articles:
                    try:
                        # EKSTRAK JUDUL YANG BERSIH
                        title = extract_title_from_search_result(article_elem)

                        if not title:
                            continue

                        # Cari link artikel
                        link_tag = article_elem.find("a", href=lambda x: x and "/read/" in x)
                        if not link_tag:
                            continue

                        article_url = link_tag["href"]

                        time.sleep(random.uniform(0.3, 0.5))

                        detail = requests.get(article_url, headers=HEADERS, timeout=10)
                        soup_detail = BeautifulSoup(detail.text, "html.parser")

                        date_str, article_year = extract_date_from_article(soup_detail)

                        if article_year != year:
                            continue

                        content = get_content(soup_detail)

                        if not content:
                            continue

                        article_data = {
                            "source": "Kompas",
                            "keyword": keyword,
                            "title": title,  # Judul sudah bersih
                            "date": date_str,
                            "year": article_year,
                            "url": article_url,
                            "content": content
                        }
                        year_data.append(article_data)
                        page_articles += 1
                        total_processed += 1

                        # TAMPILKAN JUDUL YANG SUDAH BERSIH
                        print(f"      ✓ {title[:80]} → {date_str}")

                    except Exception as e:
                        continue

                print(f"    📊 Halaman {page}: {page_articles} artikel berhasil diambil")

                time.sleep(random.uniform(1, 2))

                if page_articles == 0 and page > 1:
                    break

            except Exception as e:
                print(f"    ❌ Error: {e}")
                break

    if year_data:
        # Hapus duplikat berdasarkan URL
        unique_by_url = {}
        for item in year_data:
            if item['url'] not in unique_by_url:
                unique_by_url[item['url']] = item

        year_unique = list(unique_by_url.values())
        all_data.extend(year_unique)
        print(f"\n  ✅ TAHUN {year}: {len(year_unique)} artikel unik")
        print(f"  📝 Contoh judul di tahun {year}:")
        for i, item in enumerate(year_unique[:5]):
            print(f"     {i+1}. {item['title'][:80]}")
    else:
        print(f"\n  ⚠️ TAHUN {year}: tidak ada artikel ditemukan")

# ================================================================
# DEDUPLIKASI BERDASARKAN JUDUL
# ================================================================

print(f"\n{'='*80}")
print("📊 PROSES DEDUPLIKASI DATA")
print('='*80)

if all_data:
    df = pd.DataFrame(all_data)

    print(f"📊 Total data sebelum deduplikasi: {len(df)} artikel")

    # Deduplikasi berdasarkan judul
    df['title_lower'] = df['title'].str.lower().str.strip()
    df_unique_title = df.drop_duplicates(subset=['title_lower'], keep='first')

    duplicate_by_title = len(df) - len(df_unique_title)
    print(f"📊 Duplikat berdasarkan judul: {duplicate_by_title} artikel dihapus")

    df_final = df_unique_title.drop(columns=['title_lower'])

    # Deduplikasi berdasarkan URL
    df_final = df_final.drop_duplicates(subset=['url'], keep='first')
    duplicate_by_url = len(df_unique_title) - len(df_final)
    print(f"📊 Duplikat berdasarkan URL: {duplicate_by_url} artikel dihapus")

    print(f"📊 Total data setelah deduplikasi: {len(df_final)} artikel")

    # Urutkan
    df_final = df_final.sort_values(['year', 'date'], ascending=[False, False])

    # Simpan ke CSV
    df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n✅ File disimpan ke: {OUTPUT_FILE}")

    # ================================================================
    # STATISTIK DAN SAMPLE JUDUL
    # ================================================================

    print(f"\n{'='*80}")
    print("📊 STATISTIK DATA")
    print('='*80)

    print(f"\n📊 Statistik per tahun:")
    year_stats = df_final['year'].value_counts().sort_index(ascending=False)
    for year, count in year_stats.items():
        print(f"   {year}: {count} artikel")

    print(f"\n📊 Statistik per keyword:")
    keyword_stats = df_final['keyword'].value_counts()
    for keyword, count in keyword_stats.items():
        print(f"   {keyword}: {count} artikel")

    # TAMPILKAN JUDUL BERITA YANG SUDAH BERSIH
    print(f"\n📰 10 ARTIKEL TERBARU (judul bersih):")
    print("-" * 80)
    for i, row in df_final.head(10).iterrows():
        print(f"{i+1:2d}. [{row['year']}] {row['date']} | {row['keyword']}")
        print(f"    📌 {row['title']}")
        print(f"    🔗 {row['url'][:80]}")
        print()

    # Sample judul per tahun
    print(f"\n📰 SAMPLE JUDUL PER TAHUN (judul bersih):")
    print("-" * 80)
    for year in sorted(df_final['year'].unique(), reverse=True):
        year_df = df_final[df_final['year'] == year]
        print(f"\n📅 TAHUN {year} ({len(year_df)} artikel):")
        for i, row in year_df.head(3).iterrows():
            print(f"   • {row['title']}")
        if len(year_df) > 3:
            print(f"   ... dan {len(year_df) - 3} artikel lainnya")

    print(f"\n📁 File: {OUTPUT_FILE}")
    print(f"📊 Total artikel: {len(df_final)}")

else:
    print("\n❌ Tidak ada artikel yang berhasil di-scrape")

print(f"\n{'='*80}")
print("✅ PROSES SELESAI!")
print('='*80)


📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI
📅 Tahun: 2020
🔍 Keyword: 12 keyword

📅 TAHUN 2020

  🔍 Keyword: Perlindungan Data Pribadi
    Halaman 1: 21 artikel ditemukan
      ✓ Pemerintah Didesak Sahkan RUU Perlindungan Data Pribadi → 2020-05-06
      ✓ Pemerintah Didesak Sahkan RUU Perlindungan Data Pribadi → 2020-05-06
      ✓ RUU Perlindungan Data Pribadi Ditargetkan Selesai Oktober 2020 → 2020-07-28
      ✓ Soal RUU Perlindungan Data Pribadi, "Bola" Kini Ada di DPR → 2020-01-28
      ✓ Menkoinfo Berharap DPR Segera Bahas Draft RUU Perlindungan Data Pribadi → 2020-01-28
      ✓ RUU Perlindungan Data Pribadi Segera Dibahas DPR dan Pemerintah → 2020-02-05
      ✓ Wabah Covid-19, Menkominfo Berkomitmen Jaga Perlindungan Data Pribadi → 2020-04-29
      ✓ Ini 12 Poin yang Diatur dalam RUU Perlindungan Data Pribadi → 2020-02-25
      ✓ Pemerintah Serahkan Draft RUU Perlindungan Data Pribadi ke DPR → 2020-01-28
      ✓ Rapat, DPR-Kominfo Bahas DIM RUU Perlindungan Data Pribadi → 2020-1

In [ ]:
# ============================================================
# scraper_kompas_final.py
# Dengan perbaikan pengambilan judul yang lebih akurat
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re

# ================================================================
# KONFIGURASI
# ================================================================
TARGET_YEARS = [2021]
OUTPUT_FILE = f"kompas_pdp_2021.csv"
MAX_PAGES_PER_YEAR = 2

KEYWORDS = [
    "Perlindungan Data Pribadi",
    "UU Perlindungan Data Pribadi",
    "RUU Perlindungan Data Pribadi",
    "Undang-Undang Perlindungan Data Pribadi",
    "perlindungan data pribadi",
    "kebocoran data pribadi",
    "Kominfo data pribadi",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI EKSTRAK JUDUL YANG LEBIH AKURAT
# ================================================================

def extract_title_from_search_result(article_element):
    """Extract clean title from search result article element"""

    # Method 1: Cari h3 atau h2 yang berisi judul
    title_tag = article_element.find(['h3', 'h2'])
    if title_tag:
        # Ambil teks dari h3/h2, bersihkan dari newline dan spasi berlebih
        title = title_tag.get_text(strip=True)
        if title:
            return title

    # Method 2: Cari link dengan class tertentu
    link = article_element.find('a', class_=re.compile(r'title|article'))
    if link:
        title = link.get_text(strip=True)
        if title:
            return title

    # Method 3: Cari elemen dengan class yang mengandung kata 'title'
    title_elem = article_element.find(class_=re.compile(r'title', re.I))
    if title_elem:
        title = title_elem.get_text(strip=True)
        if title:
            return title

    # Method 4: Ambil dari link pertama yang mengandung /read/
    link = article_element.find('a', href=lambda x: x and '/read/' in x)
    if link:
        # Ambil teks, tapi pisahkan dengan baris baru
        full_text = link.get_text(separator='\n', strip=True)
        # Ambil baris pertama (biasanya judul)
        lines = [line.strip() for line in full_text.split('\n') if line.strip()]
        if lines:
            return lines[0]

    return None

def extract_date_from_article(soup):
    """Extract date from article"""

    # Method 1: Meta tag
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date and meta_date.get("content"):
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', meta_date["content"])
        if match:
            year, month, day = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # Method 2: div.read__time
    read_time = soup.find("div", class_="read__time")
    if read_time:
        text = read_time.get_text(strip=True)
        months = {
            "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
            "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
        }
        match = re.search(r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+(\d{4})', text)
        if match:
            day, month_name, year = match.groups()
            if month_name in months:
                month = months[month_name]
                return f"{year}-{month:02}-{int(day):02}", int(year)

    # Method 3: div.article__date
    article_date = soup.find("div", class_="article__date")
    if article_date:
        text = article_date.get_text(strip=True)
        match = re.search(r'(\d{2})/(\d{2})/(\d{4})', text)
        if match:
            day, month, year = match.groups()
            return f"{year}-{month}-{day}", int(year)

    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if not content:
            content = soup.find("article")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE
# ================================================================

all_data = []
total_processed = 0

print("\n" + "="*80)
print("📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI")
print("="*80)
print(f"📅 Tahun: {', '.join(map(str, TARGET_YEARS))}")
print(f"🔍 Keyword: {len(KEYWORDS)} keyword")
print("="*80)

for year in TARGET_YEARS:
    print(f"\n{'='*60}")
    print(f"📅 TAHUN {year}")
    print('='*60)

    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    year_data = []

    for keyword in KEYWORDS:
        print(f"\n  🔍 Keyword: {keyword}")
        encoded = quote_plus(keyword)

        for page in range(1, MAX_PAGES_PER_YEAR + 1):
            url = f"https://search.kompas.com/search?q={encoded}&site_id=all&start_date={start_date}&end_date={end_date}&page={page}"

            try:
                resp = requests.get(url, headers=HEADERS, timeout=10)
                soup = BeautifulSoup(resp.text, "html.parser")

                # Cari elemen artikel
                article_elements = soup.find_all("div", class_=re.compile(r'article', re.I))

                # Filter yang memiliki link /read/
                valid_articles = []
                for article in article_elements:
                    links = article.find_all("a", href=lambda x: x and "/read/" in x)
                    if links:
                        valid_articles.append(article)

                if not valid_articles:
                    # Coba cari link langsung
                    links = soup.find_all("a", href=lambda x: x and "/read/" in x)
                    if not links:
                        break
                    # Buat wrapper untuk setiap link
                    valid_articles = [link.parent for link in links[:10]]

                print(f"    Halaman {page}: {len(valid_articles)} artikel ditemukan")

                page_articles = 0
                for article_elem in valid_articles:
                    try:
                        # EKSTRAK JUDUL YANG BERSIH
                        title = extract_title_from_search_result(article_elem)

                        if not title:
                            continue

                        # Cari link artikel
                        link_tag = article_elem.find("a", href=lambda x: x and "/read/" in x)
                        if not link_tag:
                            continue

                        article_url = link_tag["href"]

                        time.sleep(random.uniform(0.3, 0.5))

                        detail = requests.get(article_url, headers=HEADERS, timeout=10)
                        soup_detail = BeautifulSoup(detail.text, "html.parser")

                        date_str, article_year = extract_date_from_article(soup_detail)

                        if article_year != year:
                            continue

                        content = get_content(soup_detail)

                        if not content:
                            continue

                        article_data = {
                            "source": "Kompas",
                            "keyword": keyword,
                            "title": title,  # Judul sudah bersih
                            "date": date_str,
                            "year": article_year,
                            "url": article_url,
                            "content": content
                        }
                        year_data.append(article_data)
                        page_articles += 1
                        total_processed += 1

                        # TAMPILKAN JUDUL YANG SUDAH BERSIH
                        print(f"      ✓ {title[:80]} → {date_str}")

                    except Exception as e:
                        continue

                print(f"    📊 Halaman {page}: {page_articles} artikel berhasil diambil")

                time.sleep(random.uniform(1, 2))

                if page_articles == 0 and page > 1:
                    break

            except Exception as e:
                print(f"    ❌ Error: {e}")
                break

    if year_data:
        # Hapus duplikat berdasarkan URL
        unique_by_url = {}
        for item in year_data:
            if item['url'] not in unique_by_url:
                unique_by_url[item['url']] = item

        year_unique = list(unique_by_url.values())
        all_data.extend(year_unique)
        print(f"\n  ✅ TAHUN {year}: {len(year_unique)} artikel unik")
        print(f"  📝 Contoh judul di tahun {year}:")
        for i, item in enumerate(year_unique[:5]):
            print(f"     {i+1}. {item['title'][:80]}")
    else:
        print(f"\n  ⚠️ TAHUN {year}: tidak ada artikel ditemukan")

# ================================================================
# DEDUPLIKASI BERDASARKAN JUDUL
# ================================================================

print(f"\n{'='*80}")
print("📊 PROSES DEDUPLIKASI DATA")
print('='*80)

if all_data:
    df = pd.DataFrame(all_data)

    print(f"📊 Total data sebelum deduplikasi: {len(df)} artikel")

    # Deduplikasi berdasarkan judul
    df['title_lower'] = df['title'].str.lower().str.strip()
    df_unique_title = df.drop_duplicates(subset=['title_lower'], keep='first')

    duplicate_by_title = len(df) - len(df_unique_title)
    print(f"📊 Duplikat berdasarkan judul: {duplicate_by_title} artikel dihapus")

    df_final = df_unique_title.drop(columns=['title_lower'])

    # Deduplikasi berdasarkan URL
    df_final = df_final.drop_duplicates(subset=['url'], keep='first')
    duplicate_by_url = len(df_unique_title) - len(df_final)
    print(f"📊 Duplikat berdasarkan URL: {duplicate_by_url} artikel dihapus")

    print(f"📊 Total data setelah deduplikasi: {len(df_final)} artikel")

    # Urutkan
    df_final = df_final.sort_values(['year', 'date'], ascending=[False, False])

    # Simpan ke CSV
    df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n✅ File disimpan ke: {OUTPUT_FILE}")

    # ================================================================
    # STATISTIK DAN SAMPLE JUDUL
    # ================================================================

    print(f"\n{'='*80}")
    print("📊 STATISTIK DATA")
    print('='*80)

    print(f"\n📊 Statistik per tahun:")
    year_stats = df_final['year'].value_counts().sort_index(ascending=False)
    for year, count in year_stats.items():
        print(f"   {year}: {count} artikel")

    print(f"\n📊 Statistik per keyword:")
    keyword_stats = df_final['keyword'].value_counts()
    for keyword, count in keyword_stats.items():
        print(f"   {keyword}: {count} artikel")

    # TAMPILKAN JUDUL BERITA YANG SUDAH BERSIH
    print(f"\n📰 10 ARTIKEL TERBARU (judul bersih):")
    print("-" * 80)
    for i, row in df_final.head(10).iterrows():
        print(f"{i+1:2d}. [{row['year']}] {row['date']} | {row['keyword']}")
        print(f"    📌 {row['title']}")
        print(f"    🔗 {row['url'][:80]}")
        print()

    # Sample judul per tahun
    print(f"\n📰 SAMPLE JUDUL PER TAHUN (judul bersih):")
    print("-" * 80)
    for year in sorted(df_final['year'].unique(), reverse=True):
        year_df = df_final[df_final['year'] == year]
        print(f"\n📅 TAHUN {year} ({len(year_df)} artikel):")
        for i, row in year_df.head(3).iterrows():
            print(f"   • {row['title']}")
        if len(year_df) > 3:
            print(f"   ... dan {len(year_df) - 3} artikel lainnya")

    print(f"\n📁 File: {OUTPUT_FILE}")
    print(f"📊 Total artikel: {len(df_final)}")

else:
    print("\n❌ Tidak ada artikel yang berhasil di-scrape")

print(f"\n{'='*80}")
print("✅ PROSES SELESAI!")
print('='*80)


📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI
📅 Tahun: 2021
🔍 Keyword: 7 keyword

📅 TAHUN 2021

  🔍 Keyword: Perlindungan Data Pribadi
    Halaman 1: 21 artikel ditemukan
      ✓ Perlindungan Data Pribadi Dinilai Sudah Mendesak → 2021-03-31
      ✓ Perlindungan Data Pribadi Dinilai Sudah Mendesak → 2021-03-31
      ✓ Kebocoran Data dan Pentingnya UU Perlindungan Data Pribadi → 2021-09-08
      ✓ Kebocoran Data Terjadi Lagi, Sampai Mana RUU Perlindungan Data Pribadi? → 2021-09-03
      ✓ Kominfo: Pembahasan RUU Perlindungan Data Pribadi Menunggu Tahun Depan → 2021-12-23
      ✓ NIK Sudah Diintegrasikan, Kemendagri Nilai Perlindungan Data Pribadi Mendesak → 2021-05-07
      ✓ Jokowi: Saya Perintahkan Menkominfo Segera Tuntaskan RUU Perlindungan Data Priba → 2021-12-10
      ✓ Internet Sudah 5G, Apa Kabar RUU Perlindungan Data Pribadi? → 2021-07-02
      ✓ DPR Perpanjang Waktu Pembahasan RUU Perlindungan Data Pribadi → 2021-06-25
      ✓ DPR: Undang-Undang Perlindungan Data Pribadi Disah

In [ ]:
# ============================================================
# scraper_kompas_final.py
# Dengan perbaikan pengambilan judul yang lebih akurat
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re

# ================================================================
# KONFIGURASI
# ================================================================
TARGET_YEARS = [2022]
OUTPUT_FILE = f"kompas_pdp_2022.csv"
MAX_PAGES_PER_YEAR = 2

KEYWORDS = [
    "Perlindungan Data Pribadi",
    "UU Perlindungan Data Pribadi",
    "RUU Perlindungan Data Pribadi",
    "Undang-Undang Perlindungan Data Pribadi",
    "perlindungan data pribadi",
    "kebocoran data pribadi",
    "Kominfo data pribadi",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI EKSTRAK JUDUL YANG LEBIH AKURAT
# ================================================================

def extract_title_from_search_result(article_element):
    """Extract clean title from search result article element"""

    # Method 1: Cari h3 atau h2 yang berisi judul
    title_tag = article_element.find(['h3', 'h2'])
    if title_tag:
        # Ambil teks dari h3/h2, bersihkan dari newline dan spasi berlebih
        title = title_tag.get_text(strip=True)
        if title:
            return title

    # Method 2: Cari link dengan class tertentu
    link = article_element.find('a', class_=re.compile(r'title|article'))
    if link:
        title = link.get_text(strip=True)
        if title:
            return title

    # Method 3: Cari elemen dengan class yang mengandung kata 'title'
    title_elem = article_element.find(class_=re.compile(r'title', re.I))
    if title_elem:
        title = title_elem.get_text(strip=True)
        if title:
            return title

    # Method 4: Ambil dari link pertama yang mengandung /read/
    link = article_element.find('a', href=lambda x: x and '/read/' in x)
    if link:
        # Ambil teks, tapi pisahkan dengan baris baru
        full_text = link.get_text(separator='\n', strip=True)
        # Ambil baris pertama (biasanya judul)
        lines = [line.strip() for line in full_text.split('\n') if line.strip()]
        if lines:
            return lines[0]

    return None

def extract_date_from_article(soup):
    """Extract date from article"""

    # Method 1: Meta tag
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date and meta_date.get("content"):
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', meta_date["content"])
        if match:
            year, month, day = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # Method 2: div.read__time
    read_time = soup.find("div", class_="read__time")
    if read_time:
        text = read_time.get_text(strip=True)
        months = {
            "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
            "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
        }
        match = re.search(r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+(\d{4})', text)
        if match:
            day, month_name, year = match.groups()
            if month_name in months:
                month = months[month_name]
                return f"{year}-{month:02}-{int(day):02}", int(year)

    # Method 3: div.article__date
    article_date = soup.find("div", class_="article__date")
    if article_date:
        text = article_date.get_text(strip=True)
        match = re.search(r'(\d{2})/(\d{2})/(\d{4})', text)
        if match:
            day, month, year = match.groups()
            return f"{year}-{month}-{day}", int(year)

    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if not content:
            content = soup.find("article")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE
# ================================================================

all_data = []
total_processed = 0

print("\n" + "="*80)
print("📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI")
print("="*80)
print(f"📅 Tahun: {', '.join(map(str, TARGET_YEARS))}")
print(f"🔍 Keyword: {len(KEYWORDS)} keyword")
print("="*80)

for year in TARGET_YEARS:
    print(f"\n{'='*60}")
    print(f"📅 TAHUN {year}")
    print('='*60)

    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    year_data = []

    for keyword in KEYWORDS:
        print(f"\n  🔍 Keyword: {keyword}")
        encoded = quote_plus(keyword)

        for page in range(1, MAX_PAGES_PER_YEAR + 1):
            url = f"https://search.kompas.com/search?q={encoded}&site_id=all&start_date={start_date}&end_date={end_date}&page={page}"

            try:
                resp = requests.get(url, headers=HEADERS, timeout=10)
                soup = BeautifulSoup(resp.text, "html.parser")

                # Cari elemen artikel
                article_elements = soup.find_all("div", class_=re.compile(r'article', re.I))

                # Filter yang memiliki link /read/
                valid_articles = []
                for article in article_elements:
                    links = article.find_all("a", href=lambda x: x and "/read/" in x)
                    if links:
                        valid_articles.append(article)

                if not valid_articles:
                    # Coba cari link langsung
                    links = soup.find_all("a", href=lambda x: x and "/read/" in x)
                    if not links:
                        break
                    # Buat wrapper untuk setiap link
                    valid_articles = [link.parent for link in links[:10]]

                print(f"    Halaman {page}: {len(valid_articles)} artikel ditemukan")

                page_articles = 0
                for article_elem in valid_articles:
                    try:
                        # EKSTRAK JUDUL YANG BERSIH
                        title = extract_title_from_search_result(article_elem)

                        if not title:
                            continue

                        # Cari link artikel
                        link_tag = article_elem.find("a", href=lambda x: x and "/read/" in x)
                        if not link_tag:
                            continue

                        article_url = link_tag["href"]

                        time.sleep(random.uniform(0.3, 0.5))

                        detail = requests.get(article_url, headers=HEADERS, timeout=10)
                        soup_detail = BeautifulSoup(detail.text, "html.parser")

                        date_str, article_year = extract_date_from_article(soup_detail)

                        if article_year != year:
                            continue

                        content = get_content(soup_detail)

                        if not content:
                            continue

                        article_data = {
                            "source": "Kompas",
                            "keyword": keyword,
                            "title": title,  # Judul sudah bersih
                            "date": date_str,
                            "year": article_year,
                            "url": article_url,
                            "content": content
                        }
                        year_data.append(article_data)
                        page_articles += 1
                        total_processed += 1

                        # TAMPILKAN JUDUL YANG SUDAH BERSIH
                        print(f"      ✓ {title[:80]} → {date_str}")

                    except Exception as e:
                        continue

                print(f"    📊 Halaman {page}: {page_articles} artikel berhasil diambil")

                time.sleep(random.uniform(1, 2))

                if page_articles == 0 and page > 1:
                    break

            except Exception as e:
                print(f"    ❌ Error: {e}")
                break

    if year_data:
        # Hapus duplikat berdasarkan URL
        unique_by_url = {}
        for item in year_data:
            if item['url'] not in unique_by_url:
                unique_by_url[item['url']] = item

        year_unique = list(unique_by_url.values())
        all_data.extend(year_unique)
        print(f"\n  ✅ TAHUN {year}: {len(year_unique)} artikel unik")
        print(f"  📝 Contoh judul di tahun {year}:")
        for i, item in enumerate(year_unique[:5]):
            print(f"     {i+1}. {item['title'][:80]}")
    else:
        print(f"\n  ⚠️ TAHUN {year}: tidak ada artikel ditemukan")

# ================================================================
# DEDUPLIKASI BERDASARKAN JUDUL
# ================================================================

print(f"\n{'='*80}")
print("📊 PROSES DEDUPLIKASI DATA")
print('='*80)

if all_data:
    df = pd.DataFrame(all_data)

    print(f"📊 Total data sebelum deduplikasi: {len(df)} artikel")

    # Deduplikasi berdasarkan judul
    df['title_lower'] = df['title'].str.lower().str.strip()
    df_unique_title = df.drop_duplicates(subset=['title_lower'], keep='first')

    duplicate_by_title = len(df) - len(df_unique_title)
    print(f"📊 Duplikat berdasarkan judul: {duplicate_by_title} artikel dihapus")

    df_final = df_unique_title.drop(columns=['title_lower'])

    # Deduplikasi berdasarkan URL
    df_final = df_final.drop_duplicates(subset=['url'], keep='first')
    duplicate_by_url = len(df_unique_title) - len(df_final)
    print(f"📊 Duplikat berdasarkan URL: {duplicate_by_url} artikel dihapus")

    print(f"📊 Total data setelah deduplikasi: {len(df_final)} artikel")

    # Urutkan
    df_final = df_final.sort_values(['year', 'date'], ascending=[False, False])

    # Simpan ke CSV
    df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n✅ File disimpan ke: {OUTPUT_FILE}")

    # ================================================================
    # STATISTIK DAN SAMPLE JUDUL
    # ================================================================

    print(f"\n{'='*80}")
    print("📊 STATISTIK DATA")
    print('='*80)

    print(f"\n📊 Statistik per tahun:")
    year_stats = df_final['year'].value_counts().sort_index(ascending=False)
    for year, count in year_stats.items():
        print(f"   {year}: {count} artikel")

    print(f"\n📊 Statistik per keyword:")
    keyword_stats = df_final['keyword'].value_counts()
    for keyword, count in keyword_stats.items():
        print(f"   {keyword}: {count} artikel")

    # TAMPILKAN JUDUL BERITA YANG SUDAH BERSIH
    print(f"\n📰 10 ARTIKEL TERBARU (judul bersih):")
    print("-" * 80)
    for i, row in df_final.head(10).iterrows():
        print(f"{i+1:2d}. [{row['year']}] {row['date']} | {row['keyword']}")
        print(f"    📌 {row['title']}")
        print(f"    🔗 {row['url'][:80]}")
        print()

    # Sample judul per tahun
    print(f"\n📰 SAMPLE JUDUL PER TAHUN (judul bersih):")
    print("-" * 80)
    for year in sorted(df_final['year'].unique(), reverse=True):
        year_df = df_final[df_final['year'] == year]
        print(f"\n📅 TAHUN {year} ({len(year_df)} artikel):")
        for i, row in year_df.head(3).iterrows():
            print(f"   • {row['title']}")
        if len(year_df) > 3:
            print(f"   ... dan {len(year_df) - 3} artikel lainnya")

    print(f"\n📁 File: {OUTPUT_FILE}")
    print(f"📊 Total artikel: {len(df_final)}")

else:
    print("\n❌ Tidak ada artikel yang berhasil di-scrape")

print(f"\n{'='*80}")
print("✅ PROSES SELESAI!")
print('='*80)


📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI
📅 Tahun: 2022
🔍 Keyword: 7 keyword

📅 TAHUN 2022

  🔍 Keyword: Perlindungan Data Pribadi
    Halaman 1: 21 artikel ditemukan
      ✓ "Differential Privacy",  Metode Efektif Perlindungan Data Pribadi → 2022-09-30
      ✓ "Differential Privacy",  Metode Efektif Perlindungan Data Pribadi → 2022-09-30
      ✓ Menanti Kelahiran Lembaga Perlindungan Data Pribadi → 2022-10-12
      ✓ Pengertian Information Privacy dan Perlindungan Data Pribadi → 2022-06-24
      ✓ Penyelesaian Sengketa dalam UU Perlindungan Data Pribadi → 2022-09-27
      ✓ Tinjauan Penerapan UU Perlindungan Data Pribadi → 2022-09-27
      ✓ Siapa Bertanggung Jawab jika Terjadi Kegagalan Perlindungan Data Pribadi → 2022-11-03
      ✓ UU PDP Disahkan, Lembaga Pengawas Perlindungan Data Pribadi Disorot → 2022-09-20
      ✓ Perlindungan Data Pribadi Akan Jadi Perhatian pada Pemilu 2024 → 2022-12-01
      ✓ Tafsir UU Perlindungan Data Pribadi yang Perlu Diketahui → 2022-09-25
      ✓

In [ ]:
# ============================================================
# scraper_kompas_final.py
# Dengan perbaikan pengambilan judul yang lebih akurat
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re

# ================================================================
# KONFIGURASI
# ================================================================
TARGET_YEARS = [2023]
OUTPUT_FILE = f"kompas_pdp_2023.csv"
MAX_PAGES_PER_YEAR = 2

KEYWORDS = [
    "Perlindungan Data Pribadi",
    "UU Perlindungan Data Pribadi",
    "RUU Perlindungan Data Pribadi",
    "Undang-Undang Perlindungan Data Pribadi",
    "perlindungan data pribadi",
    "kebocoran data pribadi",
    "Kominfo data pribadi",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI EKSTRAK JUDUL YANG LEBIH AKURAT
# ================================================================

def extract_title_from_search_result(article_element):
    """Extract clean title from search result article element"""

    # Method 1: Cari h3 atau h2 yang berisi judul
    title_tag = article_element.find(['h3', 'h2'])
    if title_tag:
        # Ambil teks dari h3/h2, bersihkan dari newline dan spasi berlebih
        title = title_tag.get_text(strip=True)
        if title:
            return title

    # Method 2: Cari link dengan class tertentu
    link = article_element.find('a', class_=re.compile(r'title|article'))
    if link:
        title = link.get_text(strip=True)
        if title:
            return title

    # Method 3: Cari elemen dengan class yang mengandung kata 'title'
    title_elem = article_element.find(class_=re.compile(r'title', re.I))
    if title_elem:
        title = title_elem.get_text(strip=True)
        if title:
            return title

    # Method 4: Ambil dari link pertama yang mengandung /read/
    link = article_element.find('a', href=lambda x: x and '/read/' in x)
    if link:
        # Ambil teks, tapi pisahkan dengan baris baru
        full_text = link.get_text(separator='\n', strip=True)
        # Ambil baris pertama (biasanya judul)
        lines = [line.strip() for line in full_text.split('\n') if line.strip()]
        if lines:
            return lines[0]

    return None

def extract_date_from_article(soup):
    """Extract date from article"""

    # Method 1: Meta tag
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date and meta_date.get("content"):
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', meta_date["content"])
        if match:
            year, month, day = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # Method 2: div.read__time
    read_time = soup.find("div", class_="read__time")
    if read_time:
        text = read_time.get_text(strip=True)
        months = {
            "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
            "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
        }
        match = re.search(r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+(\d{4})', text)
        if match:
            day, month_name, year = match.groups()
            if month_name in months:
                month = months[month_name]
                return f"{year}-{month:02}-{int(day):02}", int(year)

    # Method 3: div.article__date
    article_date = soup.find("div", class_="article__date")
    if article_date:
        text = article_date.get_text(strip=True)
        match = re.search(r'(\d{2})/(\d{2})/(\d{4})', text)
        if match:
            day, month, year = match.groups()
            return f"{year}-{month}-{day}", int(year)

    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if not content:
            content = soup.find("article")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE
# ================================================================

all_data = []
total_processed = 0

print("\n" + "="*80)
print("📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI")
print("="*80)
print(f"📅 Tahun: {', '.join(map(str, TARGET_YEARS))}")
print(f"🔍 Keyword: {len(KEYWORDS)} keyword")
print("="*80)

for year in TARGET_YEARS:
    print(f"\n{'='*60}")
    print(f"📅 TAHUN {year}")
    print('='*60)

    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    year_data = []

    for keyword in KEYWORDS:
        print(f"\n  🔍 Keyword: {keyword}")
        encoded = quote_plus(keyword)

        for page in range(1, MAX_PAGES_PER_YEAR + 1):
            url = f"https://search.kompas.com/search?q={encoded}&site_id=all&start_date={start_date}&end_date={end_date}&page={page}"

            try:
                resp = requests.get(url, headers=HEADERS, timeout=10)
                soup = BeautifulSoup(resp.text, "html.parser")

                # Cari elemen artikel
                article_elements = soup.find_all("div", class_=re.compile(r'article', re.I))

                # Filter yang memiliki link /read/
                valid_articles = []
                for article in article_elements:
                    links = article.find_all("a", href=lambda x: x and "/read/" in x)
                    if links:
                        valid_articles.append(article)

                if not valid_articles:
                    # Coba cari link langsung
                    links = soup.find_all("a", href=lambda x: x and "/read/" in x)
                    if not links:
                        break
                    # Buat wrapper untuk setiap link
                    valid_articles = [link.parent for link in links[:10]]

                print(f"    Halaman {page}: {len(valid_articles)} artikel ditemukan")

                page_articles = 0
                for article_elem in valid_articles:
                    try:
                        # EKSTRAK JUDUL YANG BERSIH
                        title = extract_title_from_search_result(article_elem)

                        if not title:
                            continue

                        # Cari link artikel
                        link_tag = article_elem.find("a", href=lambda x: x and "/read/" in x)
                        if not link_tag:
                            continue

                        article_url = link_tag["href"]

                        time.sleep(random.uniform(0.3, 0.5))

                        detail = requests.get(article_url, headers=HEADERS, timeout=10)
                        soup_detail = BeautifulSoup(detail.text, "html.parser")

                        date_str, article_year = extract_date_from_article(soup_detail)

                        if article_year != year:
                            continue

                        content = get_content(soup_detail)

                        if not content:
                            continue

                        article_data = {
                            "source": "Kompas",
                            "keyword": keyword,
                            "title": title,  # Judul sudah bersih
                            "date": date_str,
                            "year": article_year,
                            "url": article_url,
                            "content": content
                        }
                        year_data.append(article_data)
                        page_articles += 1
                        total_processed += 1

                        # TAMPILKAN JUDUL YANG SUDAH BERSIH
                        print(f"      ✓ {title[:80]} → {date_str}")

                    except Exception as e:
                        continue

                print(f"    📊 Halaman {page}: {page_articles} artikel berhasil diambil")

                time.sleep(random.uniform(1, 2))

                if page_articles == 0 and page > 1:
                    break

            except Exception as e:
                print(f"    ❌ Error: {e}")
                break

    if year_data:
        # Hapus duplikat berdasarkan URL
        unique_by_url = {}
        for item in year_data:
            if item['url'] not in unique_by_url:
                unique_by_url[item['url']] = item

        year_unique = list(unique_by_url.values())
        all_data.extend(year_unique)
        print(f"\n  ✅ TAHUN {year}: {len(year_unique)} artikel unik")
        print(f"  📝 Contoh judul di tahun {year}:")
        for i, item in enumerate(year_unique[:5]):
            print(f"     {i+1}. {item['title'][:80]}")
    else:
        print(f"\n  ⚠️ TAHUN {year}: tidak ada artikel ditemukan")

# ================================================================
# DEDUPLIKASI BERDASARKAN JUDUL
# ================================================================

print(f"\n{'='*80}")
print("📊 PROSES DEDUPLIKASI DATA")
print('='*80)

if all_data:
    df = pd.DataFrame(all_data)

    print(f"📊 Total data sebelum deduplikasi: {len(df)} artikel")

    # Deduplikasi berdasarkan judul
    df['title_lower'] = df['title'].str.lower().str.strip()
    df_unique_title = df.drop_duplicates(subset=['title_lower'], keep='first')

    duplicate_by_title = len(df) - len(df_unique_title)
    print(f"📊 Duplikat berdasarkan judul: {duplicate_by_title} artikel dihapus")

    df_final = df_unique_title.drop(columns=['title_lower'])

    # Deduplikasi berdasarkan URL
    df_final = df_final.drop_duplicates(subset=['url'], keep='first')
    duplicate_by_url = len(df_unique_title) - len(df_final)
    print(f"📊 Duplikat berdasarkan URL: {duplicate_by_url} artikel dihapus")

    print(f"📊 Total data setelah deduplikasi: {len(df_final)} artikel")

    # Urutkan
    df_final = df_final.sort_values(['year', 'date'], ascending=[False, False])

    # Simpan ke CSV
    df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n✅ File disimpan ke: {OUTPUT_FILE}")

    # ================================================================
    # STATISTIK DAN SAMPLE JUDUL
    # ================================================================

    print(f"\n{'='*80}")
    print("📊 STATISTIK DATA")
    print('='*80)

    print(f"\n📊 Statistik per tahun:")
    year_stats = df_final['year'].value_counts().sort_index(ascending=False)
    for year, count in year_stats.items():
        print(f"   {year}: {count} artikel")

    print(f"\n📊 Statistik per keyword:")
    keyword_stats = df_final['keyword'].value_counts()
    for keyword, count in keyword_stats.items():
        print(f"   {keyword}: {count} artikel")

    # TAMPILKAN JUDUL BERITA YANG SUDAH BERSIH
    print(f"\n📰 10 ARTIKEL TERBARU (judul bersih):")
    print("-" * 80)
    for i, row in df_final.head(10).iterrows():
        print(f"{i+1:2d}. [{row['year']}] {row['date']} | {row['keyword']}")
        print(f"    📌 {row['title']}")
        print(f"    🔗 {row['url'][:80]}")
        print()

    # Sample judul per tahun
    print(f"\n📰 SAMPLE JUDUL PER TAHUN (judul bersih):")
    print("-" * 80)
    for year in sorted(df_final['year'].unique(), reverse=True):
        year_df = df_final[df_final['year'] == year]
        print(f"\n📅 TAHUN {year} ({len(year_df)} artikel):")
        for i, row in year_df.head(3).iterrows():
            print(f"   • {row['title']}")
        if len(year_df) > 3:
            print(f"   ... dan {len(year_df) - 3} artikel lainnya")

    print(f"\n📁 File: {OUTPUT_FILE}")
    print(f"📊 Total artikel: {len(df_final)}")

else:
    print("\n❌ Tidak ada artikel yang berhasil di-scrape")

print(f"\n{'='*80}")
print("✅ PROSES SELESAI!")
print('='*80)


📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI
📅 Tahun: 2023
🔍 Keyword: 7 keyword

📅 TAHUN 2023

  🔍 Keyword: Perlindungan Data Pribadi
    Halaman 1: 21 artikel ditemukan
      ✓ Perlindungan Data Pribadi Perlu Diperkuat → 2023-02-06
      ✓ Perlindungan Data Pribadi Perlu Diperkuat → 2023-02-06
      ✓ Perlindungan Hukum bagi Korban Penyebarluasan Data Pribadi → 2023-02-11
      ✓ UU Perlindungan Data Pribadi: Jenis Data dan Sanksi Pidananya → 2023-07-18
      ✓ CDO, CPO, DPO, dan Masa Transisi Perlindungan Data Pribadi Korporasi → 2023-07-20
      ✓ Perlindungan Data Pribadi Online bagi Anak dan Penyandang Disabilitas → 2023-03-14
      ✓ Pemerintah Akan Buat KTP Digital, Pakar Tekankan Pentingnya Perlindungan Data Pr → 2023-02-11
      ✓ Upaya GoTo Tingkatkan Perlindungan Data Pengguna → 2023-04-04
      ✓ Waspada Penyebaran Data Pribadi dari Joki Pinjol → 2023-11-02
      ✓ Kesadaran soal Pelindungan Data Pribadi Perlu Ditingkatkan → 2023-02-02
      ✓ Pengertian dan Jenis-jenis D

In [ ]:
# ============================================================
# scraper_kompas_final.py
# Dengan perbaikan pengambilan judul yang lebih akurat
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re

# ================================================================
# KONFIGURASI
# ================================================================
TARGET_YEARS = [2024]
OUTPUT_FILE = f"kompas_pdp_2024.csv"
MAX_PAGES_PER_YEAR = 2

KEYWORDS = [
    "Perlindungan Data Pribadi",
    "UU Perlindungan Data Pribadi",
    "RUU Perlindungan Data Pribadi",
    "Undang-Undang Perlindungan Data Pribadi",
    "perlindungan data pribadi",
    "kebocoran data pribadi",
    "Kominfo data pribadi",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI EKSTRAK JUDUL YANG LEBIH AKURAT
# ================================================================

def extract_title_from_search_result(article_element):
    """Extract clean title from search result article element"""

    # Method 1: Cari h3 atau h2 yang berisi judul
    title_tag = article_element.find(['h3', 'h2'])
    if title_tag:
        # Ambil teks dari h3/h2, bersihkan dari newline dan spasi berlebih
        title = title_tag.get_text(strip=True)
        if title:
            return title

    # Method 2: Cari link dengan class tertentu
    link = article_element.find('a', class_=re.compile(r'title|article'))
    if link:
        title = link.get_text(strip=True)
        if title:
            return title

    # Method 3: Cari elemen dengan class yang mengandung kata 'title'
    title_elem = article_element.find(class_=re.compile(r'title', re.I))
    if title_elem:
        title = title_elem.get_text(strip=True)
        if title:
            return title

    # Method 4: Ambil dari link pertama yang mengandung /read/
    link = article_element.find('a', href=lambda x: x and '/read/' in x)
    if link:
        # Ambil teks, tapi pisahkan dengan baris baru
        full_text = link.get_text(separator='\n', strip=True)
        # Ambil baris pertama (biasanya judul)
        lines = [line.strip() for line in full_text.split('\n') if line.strip()]
        if lines:
            return lines[0]

    return None

def extract_date_from_article(soup):
    """Extract date from article"""

    # Method 1: Meta tag
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date and meta_date.get("content"):
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', meta_date["content"])
        if match:
            year, month, day = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # Method 2: div.read__time
    read_time = soup.find("div", class_="read__time")
    if read_time:
        text = read_time.get_text(strip=True)
        months = {
            "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
            "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
        }
        match = re.search(r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+(\d{4})', text)
        if match:
            day, month_name, year = match.groups()
            if month_name in months:
                month = months[month_name]
                return f"{year}-{month:02}-{int(day):02}", int(year)

    # Method 3: div.article__date
    article_date = soup.find("div", class_="article__date")
    if article_date:
        text = article_date.get_text(strip=True)
        match = re.search(r'(\d{2})/(\d{2})/(\d{4})', text)
        if match:
            day, month, year = match.groups()
            return f"{year}-{month}-{day}", int(year)

    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if not content:
            content = soup.find("article")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE
# ================================================================

all_data = []
total_processed = 0

print("\n" + "="*80)
print("📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI")
print("="*80)
print(f"📅 Tahun: {', '.join(map(str, TARGET_YEARS))}")
print(f"🔍 Keyword: {len(KEYWORDS)} keyword")
print("="*80)

for year in TARGET_YEARS:
    print(f"\n{'='*60}")
    print(f"📅 TAHUN {year}")
    print('='*60)

    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    year_data = []

    for keyword in KEYWORDS:
        print(f"\n  🔍 Keyword: {keyword}")
        encoded = quote_plus(keyword)

        for page in range(1, MAX_PAGES_PER_YEAR + 1):
            url = f"https://search.kompas.com/search?q={encoded}&site_id=all&start_date={start_date}&end_date={end_date}&page={page}"

            try:
                resp = requests.get(url, headers=HEADERS, timeout=10)
                soup = BeautifulSoup(resp.text, "html.parser")

                # Cari elemen artikel
                article_elements = soup.find_all("div", class_=re.compile(r'article', re.I))

                # Filter yang memiliki link /read/
                valid_articles = []
                for article in article_elements:
                    links = article.find_all("a", href=lambda x: x and "/read/" in x)
                    if links:
                        valid_articles.append(article)

                if not valid_articles:
                    # Coba cari link langsung
                    links = soup.find_all("a", href=lambda x: x and "/read/" in x)
                    if not links:
                        break
                    # Buat wrapper untuk setiap link
                    valid_articles = [link.parent for link in links[:10]]

                print(f"    Halaman {page}: {len(valid_articles)} artikel ditemukan")

                page_articles = 0
                for article_elem in valid_articles:
                    try:
                        # EKSTRAK JUDUL YANG BERSIH
                        title = extract_title_from_search_result(article_elem)

                        if not title:
                            continue

                        # Cari link artikel
                        link_tag = article_elem.find("a", href=lambda x: x and "/read/" in x)
                        if not link_tag:
                            continue

                        article_url = link_tag["href"]

                        time.sleep(random.uniform(0.3, 0.5))

                        detail = requests.get(article_url, headers=HEADERS, timeout=10)
                        soup_detail = BeautifulSoup(detail.text, "html.parser")

                        date_str, article_year = extract_date_from_article(soup_detail)

                        if article_year != year:
                            continue

                        content = get_content(soup_detail)

                        if not content:
                            continue

                        article_data = {
                            "source": "Kompas",
                            "keyword": keyword,
                            "title": title,  # Judul sudah bersih
                            "date": date_str,
                            "year": article_year,
                            "url": article_url,
                            "content": content
                        }
                        year_data.append(article_data)
                        page_articles += 1
                        total_processed += 1

                        # TAMPILKAN JUDUL YANG SUDAH BERSIH
                        print(f"      ✓ {title[:80]} → {date_str}")

                    except Exception as e:
                        continue

                print(f"    📊 Halaman {page}: {page_articles} artikel berhasil diambil")

                time.sleep(random.uniform(1, 2))

                if page_articles == 0 and page > 1:
                    break

            except Exception as e:
                print(f"    ❌ Error: {e}")
                break

    if year_data:
        # Hapus duplikat berdasarkan URL
        unique_by_url = {}
        for item in year_data:
            if item['url'] not in unique_by_url:
                unique_by_url[item['url']] = item

        year_unique = list(unique_by_url.values())
        all_data.extend(year_unique)
        print(f"\n  ✅ TAHUN {year}: {len(year_unique)} artikel unik")
        print(f"  📝 Contoh judul di tahun {year}:")
        for i, item in enumerate(year_unique[:5]):
            print(f"     {i+1}. {item['title'][:80]}")
    else:
        print(f"\n  ⚠️ TAHUN {year}: tidak ada artikel ditemukan")

# ================================================================
# DEDUPLIKASI BERDASARKAN JUDUL
# ================================================================

print(f"\n{'='*80}")
print("📊 PROSES DEDUPLIKASI DATA")
print('='*80)

if all_data:
    df = pd.DataFrame(all_data)

    print(f"📊 Total data sebelum deduplikasi: {len(df)} artikel")

    # Deduplikasi berdasarkan judul
    df['title_lower'] = df['title'].str.lower().str.strip()
    df_unique_title = df.drop_duplicates(subset=['title_lower'], keep='first')

    duplicate_by_title = len(df) - len(df_unique_title)
    print(f"📊 Duplikat berdasarkan judul: {duplicate_by_title} artikel dihapus")

    df_final = df_unique_title.drop(columns=['title_lower'])

    # Deduplikasi berdasarkan URL
    df_final = df_final.drop_duplicates(subset=['url'], keep='first')
    duplicate_by_url = len(df_unique_title) - len(df_final)
    print(f"📊 Duplikat berdasarkan URL: {duplicate_by_url} artikel dihapus")

    print(f"📊 Total data setelah deduplikasi: {len(df_final)} artikel")

    # Urutkan
    df_final = df_final.sort_values(['year', 'date'], ascending=[False, False])

    # Simpan ke CSV
    df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n✅ File disimpan ke: {OUTPUT_FILE}")

    # ================================================================
    # STATISTIK DAN SAMPLE JUDUL
    # ================================================================

    print(f"\n{'='*80}")
    print("📊 STATISTIK DATA")
    print('='*80)

    print(f"\n📊 Statistik per tahun:")
    year_stats = df_final['year'].value_counts().sort_index(ascending=False)
    for year, count in year_stats.items():
        print(f"   {year}: {count} artikel")

    print(f"\n📊 Statistik per keyword:")
    keyword_stats = df_final['keyword'].value_counts()
    for keyword, count in keyword_stats.items():
        print(f"   {keyword}: {count} artikel")

    # TAMPILKAN JUDUL BERITA YANG SUDAH BERSIH
    print(f"\n📰 10 ARTIKEL TERBARU (judul bersih):")
    print("-" * 80)
    for i, row in df_final.head(10).iterrows():
        print(f"{i+1:2d}. [{row['year']}] {row['date']} | {row['keyword']}")
        print(f"    📌 {row['title']}")
        print(f"    🔗 {row['url'][:80]}")
        print()

    # Sample judul per tahun
    print(f"\n📰 SAMPLE JUDUL PER TAHUN (judul bersih):")
    print("-" * 80)
    for year in sorted(df_final['year'].unique(), reverse=True):
        year_df = df_final[df_final['year'] == year]
        print(f"\n📅 TAHUN {year} ({len(year_df)} artikel):")
        for i, row in year_df.head(3).iterrows():
            print(f"   • {row['title']}")
        if len(year_df) > 3:
            print(f"   ... dan {len(year_df) - 3} artikel lainnya")

    print(f"\n📁 File: {OUTPUT_FILE}")
    print(f"📊 Total artikel: {len(df_final)}")

else:
    print("\n❌ Tidak ada artikel yang berhasil di-scrape")

print(f"\n{'='*80}")
print("✅ PROSES SELESAI!")
print('='*80)


📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI
📅 Tahun: 2024
🔍 Keyword: 7 keyword

📅 TAHUN 2024

  🔍 Keyword: Perlindungan Data Pribadi
    Halaman 1: 21 artikel ditemukan
      ✓ BNI Ingatkan Mahasiswa tentang Pentingnya Perlindungan Data Pribadi → 2024-10-16
      ✓ BNI Ingatkan Mahasiswa tentang Pentingnya Perlindungan Data Pribadi → 2024-10-16
      ✓ Indonesia Kekurangan Jutaan Talenta Digital, Terutama Ahli Perlindungan Data Pri → 2024-10-04
      ✓ LPS Minta Perbankan Tingkatkan Perlindungan Data Pribadi Nasabah → 2024-03-02
      ✓ UU Segera Berlaku, Pemerintah Diimbau Bentuk Lembaga Perlindungan Data Pribadi → 2024-09-21
      ✓ UU Perlindungan Data Pribadi Berlaku Sepenuhnya di Indonesia → 2024-10-18
      ✓ Migrasi TikTok-Tokopedia, Perlindungan Data Pribadi Harus Diutamakan → 2024-03-26
      ✓ Migrasi TikTok Shop ke Tokopedia, Perlindungan Data Pribadi Harus Diutamakan → 2024-03-25
      ✓ Data ASN Diduga Bocor, Pemerintah Diminta Segera Bentuk Lembaga Perlindungan Dat → 2

In [ ]:
# ============================================================
# scraper_kompas_final.py
# Dengan perbaikan pengambilan judul yang lebih akurat
# ============================================================

import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import re

# ================================================================
# KONFIGURASI
# ================================================================
TARGET_YEARS = [2025]
OUTPUT_FILE = f"kompas_pdp_2025.csv"
MAX_PAGES_PER_YEAR = 3

KEYWORDS = [
    "Perlindungan Data Pribadi",
    "UU Perlindungan Data Pribadi",
    "RUU Perlindungan Data Pribadi",
    "Undang-Undang Perlindungan Data Pribadi",
    "perlindungan data pribadi",
    "kebocoran data pribadi",
    "Kominfo data pribadi",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ================================================================
# FUNGSI EKSTRAK JUDUL YANG LEBIH AKURAT
# ================================================================

def extract_title_from_search_result(article_element):
    """Extract clean title from search result article element"""

    # Method 1: Cari h3 atau h2 yang berisi judul
    title_tag = article_element.find(['h3', 'h2'])
    if title_tag:
        # Ambil teks dari h3/h2, bersihkan dari newline dan spasi berlebih
        title = title_tag.get_text(strip=True)
        if title:
            return title

    # Method 2: Cari link dengan class tertentu
    link = article_element.find('a', class_=re.compile(r'title|article'))
    if link:
        title = link.get_text(strip=True)
        if title:
            return title

    # Method 3: Cari elemen dengan class yang mengandung kata 'title'
    title_elem = article_element.find(class_=re.compile(r'title', re.I))
    if title_elem:
        title = title_elem.get_text(strip=True)
        if title:
            return title

    # Method 4: Ambil dari link pertama yang mengandung /read/
    link = article_element.find('a', href=lambda x: x and '/read/' in x)
    if link:
        # Ambil teks, tapi pisahkan dengan baris baru
        full_text = link.get_text(separator='\n', strip=True)
        # Ambil baris pertama (biasanya judul)
        lines = [line.strip() for line in full_text.split('\n') if line.strip()]
        if lines:
            return lines[0]

    return None

def extract_date_from_article(soup):
    """Extract date from article"""

    # Method 1: Meta tag
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date and meta_date.get("content"):
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', meta_date["content"])
        if match:
            year, month, day = match.groups()
            return f"{year}-{month}-{day}", int(year)

    # Method 2: div.read__time
    read_time = soup.find("div", class_="read__time")
    if read_time:
        text = read_time.get_text(strip=True)
        months = {
            "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
            "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
        }
        match = re.search(r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+(\d{4})', text)
        if match:
            day, month_name, year = match.groups()
            if month_name in months:
                month = months[month_name]
                return f"{year}-{month:02}-{int(day):02}", int(year)

    # Method 3: div.article__date
    article_date = soup.find("div", class_="article__date")
    if article_date:
        text = article_date.get_text(strip=True)
        match = re.search(r'(\d{2})/(\d{2})/(\d{4})', text)
        if match:
            day, month, year = match.groups()
            return f"{year}-{month}-{day}", int(year)

    return None, None

def get_content(soup):
    """Ambil isi artikel"""
    try:
        content = soup.find("div", class_="read__content")
        if not content:
            content = soup.find("article")
        if content:
            paragraphs = content.find_all("p")
            return " ".join(p.text.strip() for p in paragraphs if p.text.strip())
    except:
        pass
    return ""

# ================================================================
# SCRAPE
# ================================================================

all_data = []
total_processed = 0

print("\n" + "="*80)
print("📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI")
print("="*80)
print(f"📅 Tahun: {', '.join(map(str, TARGET_YEARS))}")
print(f"🔍 Keyword: {len(KEYWORDS)} keyword")
print("="*80)

for year in TARGET_YEARS:
    print(f"\n{'='*60}")
    print(f"📅 TAHUN {year}")
    print('='*60)

    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    year_data = []

    for keyword in KEYWORDS:
        print(f"\n  🔍 Keyword: {keyword}")
        encoded = quote_plus(keyword)

        for page in range(1, MAX_PAGES_PER_YEAR + 1):
            url = f"https://search.kompas.com/search?q={encoded}&site_id=all&start_date={start_date}&end_date={end_date}&page={page}"

            try:
                resp = requests.get(url, headers=HEADERS, timeout=10)
                soup = BeautifulSoup(resp.text, "html.parser")

                # Cari elemen artikel
                article_elements = soup.find_all("div", class_=re.compile(r'article', re.I))

                # Filter yang memiliki link /read/
                valid_articles = []
                for article in article_elements:
                    links = article.find_all("a", href=lambda x: x and "/read/" in x)
                    if links:
                        valid_articles.append(article)

                if not valid_articles:
                    # Coba cari link langsung
                    links = soup.find_all("a", href=lambda x: x and "/read/" in x)
                    if not links:
                        break
                    # Buat wrapper untuk setiap link
                    valid_articles = [link.parent for link in links[:10]]

                print(f"    Halaman {page}: {len(valid_articles)} artikel ditemukan")

                page_articles = 0
                for article_elem in valid_articles:
                    try:
                        # EKSTRAK JUDUL YANG BERSIH
                        title = extract_title_from_search_result(article_elem)

                        if not title:
                            continue

                        # Cari link artikel
                        link_tag = article_elem.find("a", href=lambda x: x and "/read/" in x)
                        if not link_tag:
                            continue

                        article_url = link_tag["href"]

                        time.sleep(random.uniform(0.3, 0.5))

                        detail = requests.get(article_url, headers=HEADERS, timeout=10)
                        soup_detail = BeautifulSoup(detail.text, "html.parser")

                        date_str, article_year = extract_date_from_article(soup_detail)

                        if article_year != year:
                            continue

                        content = get_content(soup_detail)

                        if not content:
                            continue

                        article_data = {
                            "source": "Kompas",
                            "keyword": keyword,
                            "title": title,  # Judul sudah bersih
                            "date": date_str,
                            "year": article_year,
                            "url": article_url,
                            "content": content
                        }
                        year_data.append(article_data)
                        page_articles += 1
                        total_processed += 1

                        # TAMPILKAN JUDUL YANG SUDAH BERSIH
                        print(f"      ✓ {title[:80]} → {date_str}")

                    except Exception as e:
                        continue

                print(f"    📊 Halaman {page}: {page_articles} artikel berhasil diambil")

                time.sleep(random.uniform(1, 2))

                if page_articles == 0 and page > 1:
                    break

            except Exception as e:
                print(f"    ❌ Error: {e}")
                break

    if year_data:
        # Hapus duplikat berdasarkan URL
        unique_by_url = {}
        for item in year_data:
            if item['url'] not in unique_by_url:
                unique_by_url[item['url']] = item

        year_unique = list(unique_by_url.values())
        all_data.extend(year_unique)
        print(f"\n  ✅ TAHUN {year}: {len(year_unique)} artikel unik")
        print(f"  📝 Contoh judul di tahun {year}:")
        for i, item in enumerate(year_unique[:5]):
            print(f"     {i+1}. {item['title'][:80]}")
    else:
        print(f"\n  ⚠️ TAHUN {year}: tidak ada artikel ditemukan")

# ================================================================
# DEDUPLIKASI BERDASARKAN JUDUL
# ================================================================

print(f"\n{'='*80}")
print("📊 PROSES DEDUPLIKASI DATA")
print('='*80)

if all_data:
    df = pd.DataFrame(all_data)

    print(f"📊 Total data sebelum deduplikasi: {len(df)} artikel")

    # Deduplikasi berdasarkan judul
    df['title_lower'] = df['title'].str.lower().str.strip()
    df_unique_title = df.drop_duplicates(subset=['title_lower'], keep='first')

    duplicate_by_title = len(df) - len(df_unique_title)
    print(f"📊 Duplikat berdasarkan judul: {duplicate_by_title} artikel dihapus")

    df_final = df_unique_title.drop(columns=['title_lower'])

    # Deduplikasi berdasarkan URL
    df_final = df_final.drop_duplicates(subset=['url'], keep='first')
    duplicate_by_url = len(df_unique_title) - len(df_final)
    print(f"📊 Duplikat berdasarkan URL: {duplicate_by_url} artikel dihapus")

    print(f"📊 Total data setelah deduplikasi: {len(df_final)} artikel")

    # Urutkan
    df_final = df_final.sort_values(['year', 'date'], ascending=[False, False])

    # Simpan ke CSV
    df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n✅ File disimpan ke: {OUTPUT_FILE}")

    # ================================================================
    # STATISTIK DAN SAMPLE JUDUL
    # ================================================================

    print(f"\n{'='*80}")
    print("📊 STATISTIK DATA")
    print('='*80)

    print(f"\n📊 Statistik per tahun:")
    year_stats = df_final['year'].value_counts().sort_index(ascending=False)
    for year, count in year_stats.items():
        print(f"   {year}: {count} artikel")

    print(f"\n📊 Statistik per keyword:")
    keyword_stats = df_final['keyword'].value_counts()
    for keyword, count in keyword_stats.items():
        print(f"   {keyword}: {count} artikel")

    # TAMPILKAN JUDUL BERITA YANG SUDAH BERSIH
    print(f"\n📰 10 ARTIKEL TERBARU (judul bersih):")
    print("-" * 80)
    for i, row in df_final.head(10).iterrows():
        print(f"{i+1:2d}. [{row['year']}] {row['date']} | {row['keyword']}")
        print(f"    📌 {row['title']}")
        print(f"    🔗 {row['url'][:80]}")
        print()

    # Sample judul per tahun
    print(f"\n📰 SAMPLE JUDUL PER TAHUN (judul bersih):")
    print("-" * 80)
    for year in sorted(df_final['year'].unique(), reverse=True):
        year_df = df_final[df_final['year'] == year]
        print(f"\n📅 TAHUN {year} ({len(year_df)} artikel):")
        for i, row in year_df.head(3).iterrows():
            print(f"   • {row['title']}")
        if len(year_df) > 3:
            print(f"   ... dan {len(year_df) - 3} artikel lainnya")

    print(f"\n📁 File: {OUTPUT_FILE}")
    print(f"📊 Total artikel: {len(df_final)}")

else:
    print("\n❌ Tidak ada artikel yang berhasil di-scrape")

print(f"\n{'='*80}")
print("✅ PROSES SELESAI!")
print('='*80)


📰 SCRAPING KOMPAS - PERLINDUNGAN DATA PRIBADI
📅 Tahun: 2025
🔍 Keyword: 7 keyword

📅 TAHUN 2025

  🔍 Keyword: Perlindungan Data Pribadi
    Halaman 1: 20 artikel ditemukan
      ✓ Pemerintah Tegaskan Transfer Data Pribadi ke AS Sesuai UU Perlindungan Data Prib → 2025-07-23
      ✓ Pemerintah Tegaskan Transfer Data Pribadi ke AS Sesuai UU Perlindungan Data Prib → 2025-07-23
      ✓ Mengenal Jenis-Jenis Data Pribadi dan Pentingnya Perlindungan di Era Digital → 2025-07-25
      ✓ Aturan Paylater Terbit, OJK Atur Penyelenggara, Mekanisme Penagihan, hingga Perl → 2025-12-24
      ✓ Puan Minta Pemerintah Jamin Perlindungan Data Pribadi Warga Saat Bertukar dengan → 2025-07-24
      ✓ Suara Aktivis Dirisak di Dunia Digital, Menagih Janji Lembaga Perlindungan Data  → 2025-07-23
      ✓ UGM Buka Suara Soal Sidang KIP Tentang Ijazah Jokowi, Singgung Perlindungan Data → 2025-11-20
      ✓ Legislator PKS: Jangan Setujui Transfer Data Pribadi ke AS Tanpa Perlindungan → 2025-07-25
      ✓ Tips Keaman